In [ ]:
# ======================================================================================
# COMPETITIVE OFFER REPORT PIPELINE V3 - CONTEXT-FIRST BRONZE / SILVER / GOLD
# ======================================================================================
# Single-cell Colab Enterprise notebook.
#
# What changed in V3:
#   1. Bronze is no longer only raw slide text. It stores slide pages, lines, spans,
#      regions, detected tables, table cells, raw value mentions, and raw agent output.
#   2. Gemini Vision is used as a table-cell reconstruction agent, not as a final
#      offer-normalization agent. It receives a slide context pack.
#   3. Silver converts blocks/cells into offer candidates, offer values, normalized
#      offers, device bridges, and price-grid rows.
#   4. Gold has section-level business tables for easier querying:
#        - gold_decks
#        - gold_ciHeadlines
#        - gold_majorOfferChanges
#        - gold_offers
#        - gold_offerValues
#        - gold_offerDeviceBridge
#        - gold_devicePriceGrid
#        - gold_retailReadout
#
# Design principle:
#   Do NOT create one physical table per slide. Keep one table per entity type and carry
#   page_num + section_id everywhere. This gives slide-level QA without table sprawl.
# ======================================================================================

# ======================================================================================
# SECTION 00 - INSTALLS AND IMPORTS
# ======================================================================================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = [
    ("fitz", "PyMuPDF"),
    ("google.cloud.bigquery", "google-cloud-bigquery"),
    ("google.genai", "google-genai"),
    ("db_dtypes", "db-dtypes"),
    ("pyarrow", "pyarrow"),
    ("dateutil", "python-dateutil"),
]

for import_name, pip_name in REQUIRED_PACKAGES:
    root_name = import_name.split(".")[0]
    if importlib.util.find_spec(root_name) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import os
import re
import json
import time
import uuid
import hashlib
import traceback
import textwrap
import unicodedata
from pathlib import Path
from datetime import datetime, timezone, date
from typing import Any, Dict, List, Optional, Tuple

import fitz
import numpy as np
import pandas as pd
from dateutil import parser as date_parser
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

try:
    from google import genai
    from google.genai import types as genai_types
    GOOGLE_GENAI_AVAILABLE = True
except Exception as exc:
    GOOGLE_GENAI_AVAILABLE = False
    print("WARNING: google-genai import failed. Gemini Vision will be unavailable.")
    print(exc)

try:
    from google.colab import files as colab_files
    from google.colab import auth as colab_auth
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = None

print("Package and import section complete.")


# ======================================================================================
# SECTION 01 - CONFIGURATION
# ======================================================================================

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"

# Leave None to auto-detect dataset location. Your dataset recently detected as us-west1.
BQ_LOCATION = None

VERTEX_LOCATION = "us-central1"
GEMINI_MODEL = "gemini-2.5-pro"

PDF_PATH = None
USE_UPLOAD_WIDGET = True
USE_GEMINI_VISION = True
WRITE_TO_BQ = True

# Duplicate action: None prompts user. Or set "cancel", "replace", "append_anyway".
DUPLICATE_ACTION = None

# During development, keep rows queryable in gold with curation_status.
LOAD_PENDING_ROWS_TO_GOLD_FOR_QA = True
AUTO_APPROVE_GEMINI_ROWS = False
AUTO_APPROVE_DETERMINISTIC_ROWS = False
AUTO_APPROVE_MIN_CONFIDENCE = 0.86

RENDER_ZOOM = 2.0
PAGE_IMAGE_DIR_ROOT = "/tmp/competitive_offer_pages"
GEMINI_SLEEP_SECONDS = 0.5
MAX_GEMINI_PAGES_PER_RUN = 999
PROMPT_VERSION = "competitive_offer_v3_context_cells_2026_07"
PIPELINE_VERSION = "v3_context_first_bronze_cells_gold_sections"

PRINT_CLASSIFIER_QA = True
PRINT_CONTEXT_PACK_EXAMPLES = True
PRINT_SAMPLE_OUTPUTS = True
PRINT_GOLD_EXAMPLES = True

print("Configuration loaded.")
print(f"PROJECT_ID: {PROJECT_ID}")
print(f"DATASET_ID: {DATASET_ID}")
print(f"BQ_LOCATION: {BQ_LOCATION}")
print(f"VERTEX_LOCATION: {VERTEX_LOCATION}")
print(f"GEMINI_MODEL: {GEMINI_MODEL}")
print(f"USE_GEMINI_VISION: {USE_GEMINI_VISION}")
print(f"WRITE_TO_BQ: {WRITE_TO_BQ}")


# ======================================================================================
# SECTION 02 - TABLE NAMES AND SCHEMAS
# ======================================================================================

TABLES = {
    # Bronze
    "bronze_pdfDecks": "sdi_competitiveOffers_bronze_pdfDecks_weekly",
    "bronze_slidePages": "sdi_competitiveOffers_bronze_slidePages_weekly",
    "bronze_slideLines": "sdi_competitiveOffers_bronze_slideLines_weekly",
    "bronze_slideSpans": "sdi_competitiveOffers_bronze_slideSpans_weekly",
    "bronze_slideRegions": "sdi_competitiveOffers_bronze_slideRegions_weekly",
    "bronze_slideDetectedTables": "sdi_competitiveOffers_bronze_slideDetectedTables_weekly",
    "bronze_slideTableCells": "sdi_competitiveOffers_bronze_slideTableCells_weekly",
    "bronze_valueMentions": "sdi_competitiveOffers_bronze_valueMentions_weekly",
    "bronze_agentExtractionsRaw": "sdi_competitiveOffers_bronze_agentExtractionsRaw_weekly",

    # Silver
    "silver_tocPageMap": "sdi_competitiveOffers_silver_tocPageMap_weekly",
    "silver_slideSections": "sdi_competitiveOffers_silver_slideSections_weekly",
    "silver_slideContentBlocks": "sdi_competitiveOffers_silver_slideContentBlocks_weekly",
    "silver_offerCandidates": "sdi_competitiveOffers_silver_offerCandidates_weekly",
    "silver_offerValues": "sdi_competitiveOffers_silver_offerValues_weekly",
    "silver_normalizedOffers": "sdi_competitiveOffers_silver_normalizedOffers_weekly",
    "silver_offerDeviceBridge": "sdi_competitiveOffers_silver_offerDeviceBridge_weekly",
    "silver_priceGridRows": "sdi_competitiveOffers_silver_priceGridRows_weekly",

    # Review
    "review_extractionReview": "sdi_competitiveOffers_review_extractionReview_weekly",
    "review_reviewDecisions": "sdi_competitiveOffers_review_reviewDecisions_weekly",

    # Gold
    "gold_decks": "sdi_competitiveOffers_gold_decks_weekly",
    "gold_ciHeadlines": "sdi_competitiveOffers_gold_ciHeadlines_weekly",
    "gold_majorOfferChanges": "sdi_competitiveOffers_gold_majorOfferChanges_weekly",
    "gold_offers": "sdi_competitiveOffers_gold_offers_weekly",
    "gold_offerValues": "sdi_competitiveOffers_gold_offerValues_weekly",
    "gold_offerDeviceBridge": "sdi_competitiveOffers_gold_offerDeviceBridge_weekly",
    "gold_devicePriceGrid": "sdi_competitiveOffers_gold_devicePriceGrid_weekly",
    "gold_retailReadout": "sdi_competitiveOffers_gold_retailReadout_weekly",
}


def fqtn(table_key: str) -> str:
    return f"{PROJECT_ID}.{DATASET_ID}.{TABLES[table_key]}"


def sf(name: str, field_type: str, mode: str = "NULLABLE") -> bigquery.SchemaField:
    return bigquery.SchemaField(name, field_type, mode=mode)


COMMON = [
    sf("deck_id", "STRING"),
    sf("run_id", "STRING"),
    sf("pdf_hash", "STRING"),
    sf("pdf_name", "STRING"),
    sf("deck_week", "DATE"),
]

# Schemas are intentionally explicit enough for querying, but not over-modelled.
SCHEMAS = {
    "bronze_pdfDecks": [
        sf("deck_id", "STRING"), sf("run_id", "STRING"), sf("pdf_hash", "STRING"),
        sf("pdf_name", "STRING"), sf("deck_week", "DATE"), sf("pdf_path", "STRING"),
        sf("page_count", "INT64"), sf("file_size_bytes", "INT64"),
        sf("duplicate_action", "STRING"), sf("pipeline_version", "STRING"), sf("ingestion_ts", "TIMESTAMP"),
    ],
    "bronze_slidePages": COMMON + [
        sf("page_num", "INT64"), sf("page_label", "STRING"), sf("page_width", "FLOAT64"),
        sf("page_height", "FLOAT64"), sf("raw_text", "STRING"), sf("raw_text_len", "INT64"),
        sf("rendered_image_path", "STRING"), sf("title_header_text", "STRING"),
        sf("text_density_score", "FLOAT64"), sf("visual_density_score", "FLOAT64"), sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_slideLines": COMMON + [
        sf("page_num", "INT64"), sf("line_id", "STRING"), sf("block_no", "INT64"), sf("line_no", "INT64"),
        sf("span_count", "INT64"), sf("line_text", "STRING"), sf("line_text_norm", "STRING"),
        sf("x0", "FLOAT64"), sf("y0", "FLOAT64"), sf("x1", "FLOAT64"), sf("y1", "FLOAT64"),
        sf("font_size_max", "FLOAT64"), sf("font_size_avg", "FLOAT64"), sf("font_names", "STRING"),
        sf("is_bold", "BOOL"), sf("is_italic", "BOOL"), sf("is_top_zone", "BOOL"),
        sf("is_large_font", "BOOL"), sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_slideSpans": COMMON + [
        sf("page_num", "INT64"), sf("span_id", "STRING"), sf("block_no", "INT64"), sf("line_no", "INT64"),
        sf("span_no", "INT64"), sf("span_text", "STRING"), sf("span_text_norm", "STRING"),
        sf("x0", "FLOAT64"), sf("y0", "FLOAT64"), sf("x1", "FLOAT64"), sf("y1", "FLOAT64"),
        sf("font_size", "FLOAT64"), sf("font_name", "STRING"), sf("font_flags", "INT64"),
        sf("color_int", "INT64"), sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_slideRegions": COMMON + [
        sf("page_num", "INT64"), sf("region_id", "STRING"), sf("section_id", "STRING"),
        sf("region_type", "STRING"), sf("region_label", "STRING"), sf("x0", "FLOAT64"), sf("y0", "FLOAT64"),
        sf("x1", "FLOAT64"), sf("y1", "FLOAT64"), sf("region_text", "STRING"),
        sf("detected_by", "STRING"), sf("confidence", "FLOAT64"), sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_slideDetectedTables": COMMON + [
        sf("page_num", "INT64"), sf("table_id", "STRING"), sf("section_id", "STRING"),
        sf("layout_type", "STRING"), sf("table_type", "STRING"), sf("table_title", "STRING"),
        sf("x0", "FLOAT64"), sf("y0", "FLOAT64"), sf("x1", "FLOAT64"), sf("y1", "FLOAT64"),
        sf("detected_by", "STRING"), sf("confidence", "FLOAT64"), sf("raw_table_text", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_slideTableCells": COMMON + [
        sf("page_num", "INT64"), sf("table_id", "STRING"), sf("cell_id", "STRING"), sf("section_id", "STRING"),
        sf("layout_type", "STRING"), sf("table_type", "STRING"), sf("row_index", "INT64"), sf("column_index", "INT64"),
        sf("row_header", "STRING"), sf("column_header", "STRING"), sf("super_column_header", "STRING"),
        sf("cell_text", "STRING"), sf("cell_text_norm", "STRING"), sf("carrier_context", "STRING"),
        sf("brand_context", "STRING"), sf("product_context", "STRING"), sf("category_context", "STRING"),
        sf("x0", "FLOAT64"), sf("y0", "FLOAT64"), sf("x1", "FLOAT64"), sf("y1", "FLOAT64"),
        sf("detected_by", "STRING"), sf("confidence", "FLOAT64"), sf("needs_review", "BOOL"),
        sf("raw_evidence", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_valueMentions": COMMON + [
        sf("page_num", "INT64"), sf("source_table", "STRING"), sf("source_id", "STRING"),
        sf("section_id", "STRING"), sf("value_mention_id", "STRING"), sf("source_text", "STRING"),
        sf("value_text", "STRING"), sf("value_type", "STRING"), sf("value_num", "FLOAT64"),
        sf("value_num_min", "FLOAT64"), sf("value_num_max", "FLOAT64"), sf("currency", "STRING"),
        sf("unit", "STRING"), sf("period", "STRING"), sf("comparator", "STRING"),
        sf("context_before", "STRING"), sf("context_after", "STRING"), sf("confidence", "FLOAT64"),
        sf("created_ts", "TIMESTAMP"),
    ],
    "bronze_agentExtractionsRaw": COMMON + [
        sf("page_num", "INT64"), sf("section_id", "STRING"), sf("layout_type", "STRING"),
        sf("agent_task", "STRING"), sf("extractor_name", "STRING"), sf("extractor_model", "STRING"),
        sf("prompt_version", "STRING"), sf("context_pack_json", "STRING"), sf("prompt_text", "STRING"),
        sf("raw_json", "STRING"), sf("raw_response_text", "STRING"), sf("extraction_status", "STRING"),
        sf("error_message", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_tocPageMap": COMMON + [
        sf("toc_source_page_num", "INT64"), sf("toc_line_text", "STRING"), sf("toc_label", "STRING"),
        sf("expected_page_num", "INT64"), sf("expected_section_id", "STRING"),
        sf("expected_section_group", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_slideSections": COMMON + [
        sf("page_num", "INT64"), sf("section_id", "STRING"), sf("section_group", "STRING"),
        sf("layout_type", "STRING"), sf("requires_vision_extraction", "BOOL"),
        sf("expected_table_type", "STRING"), sf("extraction_route", "STRING"),
        sf("classifier_method", "STRING"), sf("classifier_confidence", "FLOAT64"),
        sf("matched_rule", "STRING"), sf("matched_pattern", "STRING"),
        sf("title_header_text", "STRING"), sf("title_header_lines_json", "STRING"),
        sf("toc_expected_section", "STRING"), sf("toc_mismatch_flag", "BOOL"),
        sf("classification_debug_json", "STRING"), sf("context_pack_json", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_slideContentBlocks": COMMON + [
        sf("block_id", "STRING"), sf("page_num", "INT64"), sf("section_id", "STRING"),
        sf("section_group", "STRING"), sf("layout_type", "STRING"), sf("block_type", "STRING"),
        sf("carrier", "STRING"), sf("brand", "STRING"), sf("category", "STRING"),
        sf("block_title", "STRING"), sf("block_text", "STRING"), sf("source_method", "STRING"),
        sf("confidence", "FLOAT64"), sf("needs_review", "BOOL"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_offerCandidates": COMMON + [
        sf("candidate_id", "STRING"), sf("block_id", "STRING"), sf("cell_id", "STRING"), sf("page_num", "INT64"),
        sf("section_id", "STRING"), sf("section_group", "STRING"), sf("layout_type", "STRING"),
        sf("carrier", "STRING"), sf("brand", "STRING"), sf("product_name", "STRING"),
        sf("product_category", "STRING"), sf("offer_type", "STRING"), sf("offer_summary", "STRING"),
        sf("offer_amount_text", "STRING"), sf("offer_value", "FLOAT64"), sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"), sf("customer_segment", "STRING"), sf("action_required", "STRING"),
        sf("date_start_text", "STRING"), sf("date_end_text", "STRING"), sf("condition_text", "STRING"),
        sf("raw_evidence", "STRING"), sf("extraction_method", "STRING"), sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"), sf("validation_errors_json", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_offerValues": COMMON + [
        sf("offer_value_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"),
        sf("section_id", "STRING"), sf("value_role", "STRING"), sf("value_type", "STRING"),
        sf("value_text", "STRING"), sf("value_num", "FLOAT64"), sf("value_num_min", "FLOAT64"),
        sf("value_num_max", "FLOAT64"), sf("currency", "STRING"), sf("unit", "STRING"),
        sf("period", "STRING"), sf("comparator", "STRING"), sf("raw_evidence", "STRING"),
        sf("confidence", "FLOAT64"), sf("created_ts", "TIMESTAMP"),
    ],
    "silver_normalizedOffers": COMMON + [
        sf("offer_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"),
        sf("section_id", "STRING"), sf("section_group", "STRING"), sf("layout_type", "STRING"),
        sf("carrier", "STRING"), sf("brand", "STRING"), sf("product_name", "STRING"),
        sf("product_category", "STRING"), sf("offer_type", "STRING"), sf("offer_summary", "STRING"),
        sf("primary_value_text", "STRING"), sf("primary_value_num", "FLOAT64"), sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"), sf("customer_segment", "STRING"), sf("action_required", "STRING"),
        sf("condition_text", "STRING"), sf("date_start", "DATE"), sf("date_end", "DATE"),
        sf("raw_evidence", "STRING"), sf("normalization_method", "STRING"), sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"), sf("auto_approve_flag", "BOOL"), sf("curation_status", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
    "silver_offerDeviceBridge": COMMON + [
        sf("bridge_id", "STRING"), sf("offer_id", "STRING"), sf("candidate_id", "STRING"),
        sf("page_num", "INT64"), sf("section_id", "STRING"), sf("carrier", "STRING"),
        sf("device_name", "STRING"), sf("device_brand", "STRING"), sf("device_family", "STRING"),
        sf("device_model", "STRING"), sf("raw_evidence", "STRING"), sf("confidence", "FLOAT64"),
        sf("created_ts", "TIMESTAMP"),
    ],
    "silver_priceGridRows": COMMON + [
        sf("grid_row_id", "STRING"), sf("cell_id", "STRING"), sf("page_num", "INT64"),
        sf("section_id", "STRING"), sf("section_group", "STRING"), sf("brand", "STRING"),
        sf("carrier", "STRING"), sf("device_name", "STRING"), sf("device_category", "STRING"),
        sf("retail_price", "FLOAT64"), sf("customer_scenario", "STRING"), sf("id_requirement", "STRING"),
        sf("rate_plan_requirement", "STRING"), sf("price_text", "STRING"), sf("price_value", "FLOAT64"),
        sf("savings_text", "STRING"), sf("savings_value", "FLOAT64"), sf("raw_evidence", "STRING"),
        sf("extraction_method", "STRING"), sf("confidence", "FLOAT64"), sf("needs_review", "BOOL"),
        sf("validation_errors_json", "STRING"), sf("created_ts", "TIMESTAMP"),
    ],
    "review_extractionReview": COMMON + [
        sf("review_id", "STRING"), sf("candidate_id", "STRING"), sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"), sf("section_id", "STRING"), sf("review_reason", "STRING"),
        sf("review_status", "STRING"), sf("source_table", "STRING"), sf("raw_payload_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
    "review_reviewDecisions": COMMON + [
        sf("decision_id", "STRING"), sf("candidate_id", "STRING"), sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"), sf("section_id", "STRING"), sf("decision_type", "STRING"),
        sf("decision_status", "STRING"), sf("decision_source", "STRING"), sf("decision_notes", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
}

# Gold schemas reuse a practical subset for consumption.
SCHEMAS.update({
    "gold_decks": [sf("deck_id", "STRING"), sf("pdf_hash", "STRING"), sf("pdf_name", "STRING"), sf("deck_week", "DATE"), sf("page_count", "INT64"), sf("pipeline_version", "STRING"), sf("loaded_ts", "TIMESTAMP")],
    "gold_ciHeadlines": COMMON + [sf("headline_id", "STRING"), sf("page_num", "INT64"), sf("headline_type", "STRING"), sf("lob", "STRING"), sf("carrier", "STRING"), sf("headline_rank", "INT64"), sf("headline_text", "STRING"), sf("raw_evidence", "STRING"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_majorOfferChanges": COMMON + [sf("change_id", "STRING"), sf("offer_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"), sf("lob", "STRING"), sf("carrier", "STRING"), sf("change_type", "STRING"), sf("product_name", "STRING"), sf("offer_summary", "STRING"), sf("primary_value_text", "STRING"), sf("primary_value_num", "FLOAT64"), sf("effective_start_text", "STRING"), sf("effective_end_text", "STRING"), sf("condition_text", "STRING"), sf("raw_evidence", "STRING"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_offers": COMMON + [sf("offer_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"), sf("section_id", "STRING"), sf("section_group", "STRING"), sf("lob", "STRING"), sf("carrier", "STRING"), sf("brand", "STRING"), sf("product_name", "STRING"), sf("product_category", "STRING"), sf("offer_type", "STRING"), sf("offer_summary", "STRING"), sf("primary_value_text", "STRING"), sf("primary_value_num", "FLOAT64"), sf("currency", "STRING"), sf("rate_plan_requirement", "STRING"), sf("customer_segment", "STRING"), sf("action_required", "STRING"), sf("condition_text", "STRING"), sf("date_start", "DATE"), sf("date_end", "DATE"), sf("raw_evidence", "STRING"), sf("confidence", "FLOAT64"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_offerValues": COMMON + [sf("offer_value_id", "STRING"), sf("offer_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"), sf("section_id", "STRING"), sf("value_role", "STRING"), sf("value_type", "STRING"), sf("value_text", "STRING"), sf("value_num", "FLOAT64"), sf("value_num_min", "FLOAT64"), sf("value_num_max", "FLOAT64"), sf("currency", "STRING"), sf("unit", "STRING"), sf("period", "STRING"), sf("comparator", "STRING"), sf("raw_evidence", "STRING"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_offerDeviceBridge": COMMON + [sf("bridge_id", "STRING"), sf("offer_id", "STRING"), sf("candidate_id", "STRING"), sf("page_num", "INT64"), sf("section_id", "STRING"), sf("carrier", "STRING"), sf("device_name", "STRING"), sf("device_brand", "STRING"), sf("device_family", "STRING"), sf("device_model", "STRING"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_devicePriceGrid": COMMON + [sf("grid_row_id", "STRING"), sf("page_num", "INT64"), sf("section_id", "STRING"), sf("section_group", "STRING"), sf("brand", "STRING"), sf("carrier", "STRING"), sf("device_name", "STRING"), sf("device_category", "STRING"), sf("retail_price", "FLOAT64"), sf("customer_scenario", "STRING"), sf("id_requirement", "STRING"), sf("rate_plan_requirement", "STRING"), sf("price_text", "STRING"), sf("price_value", "FLOAT64"), sf("savings_text", "STRING"), sf("savings_value", "FLOAT64"), sf("raw_evidence", "STRING"), sf("confidence", "FLOAT64"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
    "gold_retailReadout": COMMON + [sf("readout_id", "STRING"), sf("page_num", "INT64"), sf("retailer", "STRING"), sf("brand_or_carrier", "STRING"), sf("device_name", "STRING"), sf("promo_summary", "STRING"), sf("promo_value_text", "STRING"), sf("promo_value_num", "FLOAT64"), sf("source_context", "STRING"), sf("raw_evidence", "STRING"), sf("curation_status", "STRING"), sf("created_ts", "TIMESTAMP")],
})

print("Table names and schemas loaded.")

# ======================================================================================
# SECTION 03 - UTILITY FUNCTIONS AND BIGQUERY HELPERS
# ======================================================================================

def now_utc() -> datetime:
    return datetime.now(timezone.utc)


def print_header(title: str) -> None:
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)


def print_subheader(title: str) -> None:
    print("\n" + "-" * 110)
    print(title)
    print("-" * 110)


def show_df(df: pd.DataFrame, max_rows: int = 50) -> None:
    if df is None:
        print("DataFrame is None")
        return
    if df.empty:
        print("DataFrame is empty")
        return
    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))


def safe_json(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False, default=str)
    except Exception:
        return str(obj)


def normalize_for_match(text: Any) -> str:
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    s = s.replace("–", " ").replace("—", " ").replace("-", " ")
    s = s.replace("&", " and ")
    s = re.sub(r"[(){}:;,|/]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()


def clean_text_line(text: Any) -> str:
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))
    s = s.replace("\u200b", "")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def make_hash_id(prefix: str, *parts: Any, n: int = 24) -> str:
    raw = "||".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:n]
    return f"{prefix}_{digest}"


def compute_file_hash(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def safe_float(value: Any) -> Optional[float]:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    s = str(value).strip()
    if not s:
        return None
    s_norm = s.upper().replace(",", "").replace("$", "")
    if s_norm in {"FREE", "ON US", "0", "$0"}:
        return 0.0
    match = re.search(r"-?\d+(?:\.\d+)?", s_norm)
    return float(match.group(0)) if match else None


def parse_date_safe(value: Any) -> Optional[date]:
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return None
    try:
        return date_parser.parse(s, fuzzy=True).date()
    except Exception:
        return None


def is_null_value(value: Any) -> bool:
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def stringify_for_bq(value: Any) -> Optional[str]:
    if is_null_value(value):
        return None
    if isinstance(value, (dict, list, tuple)):
        return safe_json(value)
    return str(value)


def infer_deck_week_from_text_or_name(pdf_path: str, first_page_text: str) -> date:
    text = first_page_text or ""
    filename = os.path.basename(pdf_path or "")
    month_date_pattern = (
        r"\b("
        r"January|February|March|April|May|June|July|August|September|October|November|December"
        r")\s+\d{1,2},\s+\d{4}\b"
    )
    text_match = re.search(month_date_pattern, text, flags=re.IGNORECASE)
    if text_match:
        try:
            return date_parser.parse(text_match.group(0)).date()
        except Exception:
            pass
    filename_match = re.search(r"(?<!\d)(\d{6})(?!\d)", filename)
    if filename_match:
        try:
            return datetime.strptime(filename_match.group(1), "%m%d%y").date()
        except ValueError:
            pass
    return date.today()


if IN_COLAB:
    try:
        colab_auth.authenticate_user()
        print("Colab authentication complete.")
    except Exception as exc:
        print("WARNING: google.colab.auth.authenticate_user() is not supported in some Colab Enterprise sessions.")
        print("Continuing with the environment active credentials.")
        print(exc)

bq_client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client project: {bq_client.project}")


def detect_bq_location() -> str:
    global BQ_LOCATION
    if BQ_LOCATION:
        return BQ_LOCATION
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    try:
        dataset = bq_client.get_dataset(dataset_id)
        BQ_LOCATION = dataset.location
        print(f"Detected BigQuery dataset location: {BQ_LOCATION}")
    except NotFound:
        BQ_LOCATION = "US"
        print("Dataset does not exist yet. Defaulting BQ_LOCATION to US.")
    print(f"Using BQ_LOCATION: {BQ_LOCATION}")
    return BQ_LOCATION


def ensure_dataset_and_tables() -> None:
    location = detect_bq_location()
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    try:
        bq_client.get_dataset(dataset_id)
        print(f"Dataset exists: {dataset_id}")
    except NotFound:
        print(f"Creating dataset: {dataset_id} in {location}")
        bq_client.create_dataset(dataset)
    for table_key, schema in SCHEMAS.items():
        table_id = fqtn(table_key)
        try:
            table = bq_client.get_table(table_id)
            existing = {field.name for field in table.schema}
            missing = [field for field in schema if field.name not in existing]
            if missing:
                print(f"Adding {len(missing)} missing columns to {table_id}")
                table.schema = list(table.schema) + missing
                bq_client.update_table(table, ["schema"])
            else:
                print(f"Table exists: {table_id}")
        except NotFound:
            print(f"Creating table: {table_id}")
            bq_client.create_table(bigquery.Table(table_id, schema=schema))


def query_bq_df(sql: str, params: Optional[List[bigquery.ScalarQueryParameter]] = None) -> pd.DataFrame:
    job_config = None
    if params:
        job_config = bigquery.QueryJobConfig(query_parameters=params)
    return bq_client.query(sql, job_config=job_config, location=detect_bq_location()).to_dataframe()


def get_table_schema_map(table_key: str) -> Dict[str, bigquery.SchemaField]:
    table = bq_client.get_table(fqtn(table_key))
    return {field.name: field for field in table.schema}


def coerce_df_to_bq_schema(df: pd.DataFrame, table_key: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    schema_map = get_table_schema_map(table_key)
    out = df.copy()
    for col in schema_map:
        if col not in out.columns:
            out[col] = None
    out = out[list(schema_map.keys())]
    for col, field in schema_map.items():
        typ = field.field_type.upper()
        if typ == "STRING":
            out[col] = out[col].apply(stringify_for_bq)
        elif typ in {"INT64", "INTEGER"}:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
        elif typ in {"FLOAT64", "FLOAT", "NUMERIC", "BIGNUMERIC"}:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        elif typ in {"BOOL", "BOOLEAN"}:
            def to_bool(v: Any) -> Optional[bool]:
                if is_null_value(v):
                    return None
                if isinstance(v, bool):
                    return v
                s = str(v).strip().lower()
                if s in {"true", "1", "yes", "y"}:
                    return True
                if s in {"false", "0", "no", "n"}:
                    return False
                return None
            out[col] = out[col].apply(to_bool)
        elif typ == "DATE":
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.date
        elif typ == "TIMESTAMP":
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)
    return out


def append_df_to_bq(df: pd.DataFrame, table_key: str) -> None:
    if df is None or df.empty:
        print(f"Skipping empty dataframe for {table_key}")
        return
    table_id = fqtn(table_key)
    clean_df = coerce_df_to_bq_schema(df, table_key)
    if clean_df.empty:
        print(f"Skipping empty cleaned dataframe for {table_key}")
        return
    job_config = bigquery.LoadJobConfig(
        schema=bq_client.get_table(table_id).schema,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )
    print(f"Loading {len(clean_df):,} rows into {table_id}")
    job = bq_client.load_table_from_dataframe(clean_df, table_id, job_config=job_config, location=detect_bq_location())
    job.result()
    print(f"Loaded {len(clean_df):,} rows into {table_id}")


def get_existing_decks_by_hash(pdf_hash: str) -> pd.DataFrame:
    if not WRITE_TO_BQ:
        return pd.DataFrame()
    try:
        bq_client.get_table(fqtn("bronze_pdfDecks"))
    except NotFound:
        return pd.DataFrame()
    sql = f"""
    SELECT deck_id, run_id, pdf_name, deck_week, ingestion_ts, duplicate_action
    FROM `{fqtn("bronze_pdfDecks")}`
    WHERE pdf_hash = @pdf_hash
    ORDER BY ingestion_ts DESC
    """
    params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]
    return query_bq_df(sql, params)


def resolve_duplicate_action(existing_df: pd.DataFrame) -> str:
    if existing_df.empty:
        return "new"
    print_header("DUPLICATE PDF HASH FOUND")
    show_df(existing_df, max_rows=20)
    action = DUPLICATE_ACTION
    if action is None:
        action = input("Choose action: cancel / replace / append_anyway: ").strip().lower()
    allowed = {"cancel", "replace", "append_anyway"}
    if action not in allowed:
        raise ValueError(f"Invalid duplicate action {action}. Allowed: {sorted(allowed)}")
    if action == "cancel":
        raise RuntimeError("Pipeline cancelled because duplicate action was cancel.")
    print(f"Duplicate action selected: {action}")
    return action


def delete_existing_rows_for_pdf_hash(pdf_hash: str) -> None:
    print_header("REPLACE MODE - DELETING EXISTING ROWS FOR PDF HASH")
    for table_key in TABLES:
        table_id = fqtn(table_key)
        try:
            table = bq_client.get_table(table_id)
            cols = {field.name for field in table.schema}
            if "pdf_hash" not in cols:
                print(f"Skipping {table_id}. No pdf_hash column.")
                continue
            sql = f"DELETE FROM `{table_id}` WHERE pdf_hash = @pdf_hash"
            params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]
            bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params), location=detect_bq_location()).result()
            print(f"Deleted rows from {table_id}")
        except Exception as exc:
            print(f"WARNING: Delete failed for {table_id}")
            print(exc)

# ======================================================================================
# SECTION 04 - PDF INPUT, UPLOAD WIDGET, AND BRONZE EXTRACTION
# ======================================================================================

def validate_pdf_path(pdf_path: str) -> str:
    if not pdf_path:
        raise ValueError("PDF_PATH is empty. Upload a PDF or set PDF_PATH manually.")
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    if not pdf_path.lower().endswith(".pdf"):
        raise ValueError(f"Expected a PDF file, got: {pdf_path}")
    return pdf_path


def upload_pdf_with_colab_widget() -> str:
    if not IN_COLAB:
        raise RuntimeError("Upload widget is only available in Colab. Set PDF_PATH manually when not in Colab.")
    print_header("PDF UPLOAD")
    print("Upload one Competitive Offer Report PDF using the file upload widget.")
    print("After upload, the pipeline computes pdf_hash and checks BigQuery duplicates.")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No PDF was uploaded.")
    first_name = list(uploaded.keys())[0]
    candidate_paths = [f"/content/{first_name}", first_name]
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            print(f"Uploaded PDF saved to: {candidate}")
            return validate_pdf_path(candidate)
    # Colab can rename duplicates as 'file (1).pdf'. Find the newest matching PDF.
    pdfs = sorted(Path("/content").glob("*.pdf"), key=lambda p: p.stat().st_mtime, reverse=True)
    if pdfs:
        print(f"Using newest uploaded PDF: {pdfs[0]}")
        return validate_pdf_path(str(pdfs[0]))
    raise FileNotFoundError("Could not locate uploaded PDF in /content.")


def choose_pdf_path() -> str:
    print_header("PDF INPUT RESOLUTION")
    if PDF_PATH:
        print(f"Using configured PDF_PATH: {PDF_PATH}")
        return validate_pdf_path(PDF_PATH)
    if USE_UPLOAD_WIDGET:
        return upload_pdf_with_colab_widget()
    raise ValueError("Set PDF_PATH or set USE_UPLOAD_WIDGET=True.")


def render_page_to_png(page: fitz.Page, output_path: str, zoom: float) -> None:
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    pix.save(output_path)


def infer_deck_week_from_text_or_name(pdf_path: str, first_page_text: str) -> date:
    text = first_page_text or ""
    filename = os.path.basename(pdf_path or "")
    month_date_pattern = (
        r"\b("
        r"January|February|March|April|May|June|July|August|September|October|November|December"
        r")\s+\d{1,2},\s+\d{4}\b"
    )
    text_match = re.search(month_date_pattern, text, flags=re.IGNORECASE)
    if text_match:
        try:
            return date_parser.parse(text_match.group(0)).date()
        except Exception:
            pass
    filename_match = re.search(r"(?<!\d)(\d{6})(?!\d)", filename)
    if filename_match:
        try:
            return datetime.strptime(filename_match.group(1), "%m%d%y").date()
        except ValueError:
            pass
    return date.today()


def density_score_from_text(raw_text: str) -> float:
    if not raw_text:
        return 0.0
    lines = [x for x in raw_text.splitlines() if x.strip()]
    return float(len(lines))


def extract_pdf_to_bronze(pdf_path: str, deck_id: str, run_id: str, pdf_hash: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print_header("BRONZE EXTRACTION - PAGES, LINES, SPANS, AND RENDERED IMAGES")
    pdf_name = os.path.basename(pdf_path)
    image_dir = Path(PAGE_IMAGE_DIR_ROOT) / run_id
    image_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(pdf_path)
    page_rows: List[Dict[str, Any]] = []
    line_rows: List[Dict[str, Any]] = []
    span_rows: List[Dict[str, Any]] = []
    print(f"PDF: {pdf_name}")
    print(f"Pages: {doc.page_count}")
    print(f"Rendered images: {image_dir}")

    for page_index in range(doc.page_count):
        page = doc[page_index]
        page_num = page_index + 1
        rect = page.rect
        page_width = float(rect.width)
        page_height = float(rect.height)
        raw_text = page.get_text("text") or ""
        image_path = str(image_dir / f"page_{page_num:03d}.png")
        render_page_to_png(page, image_path, RENDER_ZOOM)

        page_rows.append({
            "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name,
            "deck_week": deck_week, "page_num": page_num, "page_label": str(page_num),
            "page_width": page_width, "page_height": page_height, "raw_text": raw_text,
            "raw_text_len": len(raw_text), "rendered_image_path": image_path,
            "title_header_text": None, "text_density_score": density_score_from_text(raw_text),
            "visual_density_score": None, "created_ts": now_utc(),
        })

        page_dict = page.get_text("dict")
        for block_no, block in enumerate(page_dict.get("blocks", [])):
            if block.get("type") != 0:
                continue
            for line_no, line in enumerate(block.get("lines", [])):
                spans = line.get("spans", []) or []
                if not spans:
                    continue
                text_parts: List[str] = []
                font_sizes: List[float] = []
                font_names: List[str] = []
                x0s: List[float] = []
                y0s: List[float] = []
                x1s: List[float] = []
                y1s: List[float] = []
                is_bold = False
                is_italic = False
                for span_no, span in enumerate(spans):
                    span_text = span.get("text", "") or ""
                    if span_text:
                        text_parts.append(span_text)
                    bbox = span.get("bbox")
                    if bbox:
                        x0, y0, x1, y1 = [float(v) for v in bbox]
                        x0s.append(x0); y0s.append(y0); x1s.append(x1); y1s.append(y1)
                    else:
                        x0 = y0 = x1 = y1 = None
                    font_size = float(span.get("size")) if span.get("size") is not None else None
                    font_name = span.get("font", "") or ""
                    if font_size is not None:
                        font_sizes.append(font_size)
                    if font_name:
                        font_names.append(font_name)
                        if any(k in font_name.lower() for k in ["bold", "black", "heavy"]):
                            is_bold = True
                        if any(k in font_name.lower() for k in ["italic", "oblique"]):
                            is_italic = True
                    span_id = make_hash_id("span", deck_id, page_num, block_no, line_no, span_no, span_text, x0, y0)
                    span_rows.append({
                        "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name,
                        "deck_week": deck_week, "page_num": page_num, "span_id": span_id,
                        "block_no": int(block_no), "line_no": int(line_no), "span_no": int(span_no),
                        "span_text": clean_text_line(span_text), "span_text_norm": normalize_for_match(span_text),
                        "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                        "font_size": font_size, "font_name": font_name, "font_flags": int(span.get("flags", 0) or 0),
                        "color_int": int(span.get("color", 0) or 0), "created_ts": now_utc(),
                    })
                line_text = clean_text_line(" ".join(text_parts))
                if not line_text:
                    continue
                x0_line = min(x0s) if x0s else None
                y0_line = min(y0s) if y0s else None
                x1_line = max(x1s) if x1s else None
                y1_line = max(y1s) if y1s else None
                line_id = make_hash_id("line", deck_id, page_num, block_no, line_no, line_text, x0_line, y0_line)
                line_rows.append({
                    "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name,
                    "deck_week": deck_week, "page_num": page_num, "line_id": line_id,
                    "block_no": int(block_no), "line_no": int(line_no), "span_count": int(len(spans)),
                    "line_text": line_text, "line_text_norm": normalize_for_match(line_text),
                    "x0": x0_line, "y0": y0_line, "x1": x1_line, "y1": y1_line,
                    "font_size_max": max(font_sizes) if font_sizes else None,
                    "font_size_avg": float(np.mean(font_sizes)) if font_sizes else None,
                    "font_names": ", ".join(sorted(set(font_names))), "is_bold": bool(is_bold),
                    "is_italic": bool(is_italic), "is_top_zone": bool(y0_line is not None and y0_line <= page_height * 0.30),
                    "is_large_font": False, "created_ts": now_utc(),
                })
    doc.close()
    pages_df = pd.DataFrame(page_rows)
    lines_df = pd.DataFrame(line_rows)
    spans_df = pd.DataFrame(span_rows)
    if not lines_df.empty:
        lines_df["is_large_font"] = False
        for p, g in lines_df.groupby("page_num"):
            vals = pd.to_numeric(g["font_size_max"], errors="coerce").dropna()
            if not vals.empty:
                q80 = vals.quantile(0.80)
                idx = g[pd.to_numeric(g["font_size_max"], errors="coerce") >= q80].index
                lines_df.loc[idx, "is_large_font"] = True
    print(f"bronze_slidePages rows: {len(pages_df):,}")
    print(f"bronze_slideLines rows: {len(lines_df):,}")
    print(f"bronze_slideSpans rows: {len(spans_df):,}")
    return pages_df, lines_df, spans_df

# ======================================================================================
# SECTION 05 - TOC PARSING, CLASSIFICATION, AND SLIDE CONTEXT PACKS
# ======================================================================================

SECTION_LAYOUT_MAP = {
    "cover": "cover", "confidentiality_notice": "notice", "table_of_contents": "toc",
    "section_divider_ci_headlines": "divider", "section_divider_offer_updates": "divider",
    "section_divider_postpaid": "divider", "section_divider_prepaid": "divider", "end_slide": "end_slide",
    "ci_headlines_postpaid": "narrative_headlines", "ci_headlines_prepaid": "narrative_headlines",
    "major_offer_changes": "major_change_grid",
    "postpaid_apple_device_offers": "dense_matrix", "postpaid_samsung_google_device_offers": "dense_matrix",
    "postpaid_motorola_affordable_byod_offers": "dense_matrix", "postpaid_bts_watch_offers": "dense_matrix",
    "postpaid_bts_tablet_offers": "dense_matrix", "postpaid_bts_other_offers": "dense_matrix",
    "postpaid_service_offers": "dense_text_table", "cable_mvno_handset_offers": "dense_text_table",
    "cable_mvno_service_offers": "dense_text_table", "business_device_offers": "dense_matrix",
    "business_newline_byod_offers": "dense_matrix", "national_retail_readout": "narrative_readout",
    "prepaid_brand_offers": "dense_prepaid_table", "prepaid_metro_price_grid": "price_grid",
    "prepaid_flagship_offers": "dense_prepaid_table", "walmart_prepaid_offers": "dense_prepaid_table",
    "unknown": "unknown",
}

SECTION_GROUP_MAP = {
    "ci_headlines_postpaid": "ci_headlines", "ci_headlines_prepaid": "ci_headlines",
    "major_offer_changes": "major_offer_changes",
    "postpaid_apple_device_offers": "postpaid", "postpaid_samsung_google_device_offers": "postpaid",
    "postpaid_motorola_affordable_byod_offers": "postpaid", "postpaid_bts_watch_offers": "postpaid_bts",
    "postpaid_bts_tablet_offers": "postpaid_bts", "postpaid_bts_other_offers": "postpaid_bts",
    "postpaid_service_offers": "postpaid", "cable_mvno_handset_offers": "cable_mvno",
    "cable_mvno_service_offers": "cable_mvno", "business_device_offers": "business",
    "business_newline_byod_offers": "business", "national_retail_readout": "national_retail",
    "prepaid_brand_offers": "prepaid", "prepaid_metro_price_grid": "prepaid_price_grid",
    "prepaid_flagship_offers": "prepaid", "walmart_prepaid_offers": "prepaid_retail",
}

EXPECTED_TABLE_TYPE_BY_SECTION = {
    "major_offer_changes": "major_offer_changes",
    "postpaid_apple_device_offers": "postpaid_device_matrix",
    "postpaid_samsung_google_device_offers": "postpaid_device_matrix",
    "postpaid_motorola_affordable_byod_offers": "postpaid_device_matrix",
    "postpaid_bts_watch_offers": "bts_matrix", "postpaid_bts_tablet_offers": "bts_matrix",
    "postpaid_bts_other_offers": "bts_matrix", "postpaid_service_offers": "service_table",
    "cable_mvno_handset_offers": "cable_mvno_handset_table", "cable_mvno_service_offers": "cable_mvno_service_table",
    "business_device_offers": "business_matrix", "business_newline_byod_offers": "business_matrix",
    "prepaid_brand_offers": "prepaid_brand_table", "prepaid_metro_price_grid": "metro_price_grid",
    "prepaid_flagship_offers": "prepaid_flagship_table", "walmart_prepaid_offers": "walmart_prepaid_table",
    "national_retail_readout": "retail_readout",
}

VISION_LAYOUT_TYPES = {"dense_matrix", "dense_text_table", "dense_prepaid_table", "price_grid"}

TITLE_FIRST_OVERRIDES = [
    {"rule_name": "ci_headlines_postpaid_title_override", "section_id": "ci_headlines_postpaid", "patterns": [r"\bcompetitive intelligence headlines\b.*\bpostpaid\b", r"\bcompetitive intelligence headlines\b.*\bcable mvno\b", r"\bpostpaid\b.*\bcable mvno\b.*\bkey highlights\b", r"\bpostpaid\b.*\bkey highlights\b"]},
    {"rule_name": "ci_headlines_prepaid_title_override", "section_id": "ci_headlines_prepaid", "patterns": [r"\bcompetitive intelligence headlines\b.*\bprepaid\b", r"\bprepaid\b.*\bmvno\b.*\bkey highlights\b", r"\bprepaid\b.*\bkey highlights\b"]},
    {"rule_name": "bts_watches_title_override", "section_id": "postpaid_bts_watch_offers", "patterns": [r"\bpost paid bts watches\b", r"\bpostpaid bts watches\b", r"\bbts watches\b"]},
    {"rule_name": "bts_tablet_title_override", "section_id": "postpaid_bts_tablet_offers", "patterns": [r"\bpost paid bts tablet\b", r"\bpost paid bts tablets\b", r"\bpostpaid bts tablet\b", r"\bpostpaid bts tablets\b", r"\bbts tablet\b", r"\bbts tablets\b"]},
    {"rule_name": "bts_other_title_override", "section_id": "postpaid_bts_other_offers", "patterns": [r"\bpost paid bts and other offers\b", r"\bpostpaid bts and other offers\b", r"\bbts and other offers\b"]},
    {"rule_name": "cable_mvno_service_title_override", "section_id": "cable_mvno_service_offers", "patterns": [r"\bcable mvno service pricing and offers\b", r"\bcable mvno service offers\b", r"\bcable mvno\b.*\bservice pricing\b"]},
    {"rule_name": "metro_price_grid_title_override", "section_id": "prepaid_metro_price_grid", "patterns": [r"\bmetro price grid\b", r"\bmetro by t mobile\b.*\bprice grid\b"]},
]

SLIDE_HEADER_RULES = [
    {"rule_name": "major_offer_changes", "section_id": "major_offer_changes", "patterns": [r"\bmajor offer changes\b", r"\bmajor competitive offer updates\b"]},
    {"rule_name": "postpaid_apple_device_offers", "section_id": "postpaid_apple_device_offers", "patterns": [r"\bpost paid apple smartphone offers\b", r"\bpostpaid apple smartphone offers\b", r"\bapple smartphone offers\b", r"\bapple device offers\b"]},
    {"rule_name": "postpaid_samsung_google_device_offers", "section_id": "postpaid_samsung_google_device_offers", "patterns": [r"\bpost paid samsung and google smartphone offers\b", r"\bpostpaid samsung and google smartphone offers\b", r"\bsamsung and google smartphone offers\b", r"\bsamsung\b.*\bgoogle\b.*\boffers\b"]},
    {"rule_name": "postpaid_motorola_affordable_byod_offers", "section_id": "postpaid_motorola_affordable_byod_offers", "patterns": [r"\bpost paid motorola affordable phones and byod offers\b", r"\bpostpaid motorola affordable phones and byod offers\b", r"\bmotorola affordable phones and byod offers\b"]},
    {"rule_name": "postpaid_service_offers", "section_id": "postpaid_service_offers", "patterns": [r"\bt mobile and at and t post paid service offers\b", r"\bt mobile and at and t postpaid service offers\b", r"\bverizon post paid service offers\b", r"\bverizon postpaid service offers\b", r"\bt mobile verizon at and t services\b"]},
    {"rule_name": "cable_mvno_handset_offers", "section_id": "cable_mvno_handset_offers", "patterns": [r"\bpostpaid cable mvno handset offers\b", r"\bcable mvno handset offers\b", r"\bpostpaid cable mvno\b.*\bhandset\b"]},
    {"rule_name": "business_device_offers", "section_id": "business_device_offers", "patterns": [r"\bbusiness flagship device offers\b", r"\bbusiness device offers\b"]},
    {"rule_name": "national_retail_readout", "section_id": "national_retail_readout", "patterns": [r"\bnational retail\b", r"\bpromotions marketplace\b", r"\bkey highlights week ending\b"]},
    {"rule_name": "prepaid_brand_offers", "section_id": "prepaid_brand_offers", "patterns": [r"\bprepaid brands current offers\b", r"\bboost mobile cricket wireless\b", r"\bmetro by t mobile boost mobile cricket\b"]},
    {"rule_name": "prepaid_flagship_offers", "section_id": "prepaid_flagship_offers", "patterns": [r"\bflagship brands prepaid current offers\b", r"\bt mobile prepaid at and t prepaid verizon prepaid\b"]},
    {"rule_name": "walmart_prepaid_offers", "section_id": "walmart_prepaid_offers", "patterns": [r"\bwalmart current offers\b", r"\bwalmart\b.*\bmetro\b.*\bcricket\b.*\bboost\b"]},
]

ALLOWED_CARRIERS_DEFAULT = ["T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Optimum", "Metro", "Boost", "Cricket", "Total Wireless", "Straight Talk", "Visible", "Google Fi", "Tello", "Simple Mobile", "Walmart"]


def print_active_classifier_rules() -> None:
    print_header("ACTIVE CLASSIFIER RULES")
    for title, rules in [("TITLE-FIRST OVERRIDES", TITLE_FIRST_OVERRIDES), ("HEADER REGISTRY", SLIDE_HEADER_RULES)]:
        print(title)
        for rule in rules:
            print(f"  {rule['rule_name']} -> {rule['section_id']}")
            for pattern in rule["patterns"]:
                print(f"    {pattern}")


def is_header_noise_line(line: str) -> bool:
    norm = normalize_for_match(line)
    if not norm or len(norm) <= 1:
        return True
    noise_patterns = [r"^\d+$", r"^page\s+\d+$", r"^t mobile confidential$", r"^channel mciinsights$", r"^source\b", r"^sources\b", r"^note\b", r"^notes\b", r"^reviewed updated by\b"]
    return any(re.search(pattern, norm, flags=re.IGNORECASE) for pattern in noise_patterns)


def extract_title_header_lines(page_num: int, pages_df: pd.DataFrame, lines_df: pd.DataFrame) -> Tuple[List[str], str, str]:
    raw_row = pages_df[pages_df["page_num"] == page_num].iloc[0]
    raw_text = raw_row.get("raw_text") or ""
    page_height = float(raw_row.get("page_height") or 0)
    page_lines = lines_df[lines_df["page_num"] == page_num].copy()
    candidates: List[str] = []
    if not page_lines.empty and page_height > 0:
        page_lines["sort_y"] = pd.to_numeric(page_lines["y0"], errors="coerce")
        page_lines["sort_x"] = pd.to_numeric(page_lines["x0"], errors="coerce")
        page_lines = page_lines.sort_values(["sort_y", "sort_x"])
        header_zone = page_lines[(page_lines["y0"].fillna(999999) <= page_height * 0.30) | (page_lines["is_large_font"] == True)]
        for _, lr in header_zone.head(30).iterrows():
            line = clean_text_line(lr.get("line_text"))
            if not is_header_noise_line(line) and len(line) <= 160:
                candidates.append(line)
            if len(candidates) >= 8:
                break
    if not candidates:
        for raw_line in str(raw_text).splitlines()[:18]:
            line = clean_text_line(raw_line)
            if not is_header_noise_line(line) and len(line) <= 160:
                candidates.append(line)
            if len(candidates) >= 8:
                break
    deduped: List[str] = []
    seen = set()
    for item in candidates:
        key = normalize_for_match(item)
        if key and key not in seen:
            deduped.append(item)
            seen.add(key)
    header_text = "\n".join(deduped)
    return deduped, header_text, normalize_for_match(" ".join(deduped))


def match_rules(rules: List[Dict[str, Any]], header_norm: str) -> Optional[Dict[str, str]]:
    for rule in rules:
        for pattern in rule["patterns"]:
            if re.search(pattern, header_norm, flags=re.IGNORECASE):
                return {"section_id": rule["section_id"], "rule_name": rule["rule_name"], "matched_pattern": pattern}
    return None


def priority_classify(page_num: int, raw_text: str, header_norm: str) -> Optional[Dict[str, str]]:
    full_norm = normalize_for_match(raw_text)
    raw_lines = [clean_text_line(x) for x in str(raw_text).splitlines() if clean_text_line(x)]
    if page_num == 1 and "competitive offer report" in full_norm:
        return {"section_id": "cover", "rule_name": "priority_cover", "matched_pattern": "page 1 competitive offer report"}
    if "notice of confidentiality" in full_norm or "designated t mobile confidential" in full_norm:
        return {"section_id": "confidentiality_notice", "rule_name": "priority_confidentiality_notice", "matched_pattern": "notice of confidentiality"}
    if "table of contents" in full_norm and ("ci headlines" in full_norm or "competitive offer updates" in full_norm):
        return {"section_id": "table_of_contents", "rule_name": "priority_table_of_contents", "matched_pattern": "table of contents"}
    if "ci headlines" in header_norm and len(raw_lines) <= 10:
        return {"section_id": "section_divider_ci_headlines", "rule_name": "priority_ci_headlines_divider", "matched_pattern": "sparse CI Headlines divider"}
    if "competitive offer updates" in header_norm and len(raw_lines) <= 12:
        return {"section_id": "section_divider_offer_updates", "rule_name": "priority_offer_updates_divider", "matched_pattern": "sparse Competitive Offer Updates divider"}
    if "competitive offers" in full_norm and "promotional updates" in full_norm and "postpaid" in full_norm and len(raw_lines) <= 14:
        return {"section_id": "section_divider_postpaid", "rule_name": "priority_postpaid_divider", "matched_pattern": "Competitive Offers Promotional Updates Postpaid"}
    if "competitive offers" in full_norm and "promotional updates" in full_norm and "prepaid" in full_norm and len(raw_lines) <= 14:
        return {"section_id": "section_divider_prepaid", "rule_name": "priority_prepaid_divider", "matched_pattern": "Competitive Offers Promotional Updates Prepaid"}
    meaningful_lines = []
    for line in raw_lines:
        norm = normalize_for_match(line)
        if norm in {"t mobile confidential", "t-mobile confidential", str(page_num)}:
            continue
        meaningful_lines.append(line)
    if page_num >= 25 and len(meaningful_lines) <= 2:
        return {"section_id": "end_slide", "rule_name": "priority_sparse_final_slide", "matched_pattern": "sparse final slide"}
    if len(raw_lines) <= 6 and re.search(r"\bare you with us\b|\bthank you\b|\bquestions\b", full_norm, flags=re.IGNORECASE):
        return {"section_id": "end_slide", "rule_name": "priority_end_slide", "matched_pattern": "end slide phrase"}
    return None


def infer_expected_section_from_toc_label(label: str, expected_page_num: int, page_nums: List[int]) -> Tuple[str, str]:
    label_norm = normalize_for_match(label)
    pages_sorted = sorted(set(page_nums))
    page_pos = pages_sorted.index(expected_page_num) if expected_page_num in pages_sorted else 0
    if "ci headlines" in label_norm or "competitive intelligence" in label_norm:
        return ("ci_headlines_postpaid", "ci_headlines") if page_pos == 0 else ("ci_headlines_prepaid", "ci_headlines")
    if "major competitive offer" in label_norm or "major offer" in label_norm:
        return "major_offer_changes", "major_offer_changes"
    if "apple device" in label_norm or "apple smartphone" in label_norm:
        return "postpaid_apple_device_offers", "postpaid"
    if "samsung" in label_norm or "google" in label_norm:
        return "postpaid_samsung_google_device_offers", "postpaid"
    if "2nd tier" in label_norm or "affordable device" in label_norm or "byod offers" in label_norm:
        return "postpaid_motorola_affordable_byod_offers", "postpaid"
    if "consumer bts" in label_norm or "bts offers" in label_norm:
        return ["postpaid_bts_watch_offers", "postpaid_bts_tablet_offers", "postpaid_bts_other_offers"][min(page_pos, 2)], "postpaid_bts"
    if "postpaid cable mvnos" in label_norm or "cable mvno" in label_norm:
        return ("cable_mvno_handset_offers", "cable_mvno") if page_pos == 0 else ("cable_mvno_service_offers", "cable_mvno")
    if "postpaid business" in label_norm or "business" in label_norm:
        return "business_device_offers", "business"
    if "national retail" in label_norm:
        return "national_retail_readout", "national_retail"
    if label_norm.strip() == "prepaid":
        return "section_divider_prepaid", "prepaid"
    if "prepaid brands current offers" in label_norm:
        return "prepaid_brand_offers", "prepaid"
    if "prepaid flagship" in label_norm:
        return "prepaid_flagship_offers", "prepaid"
    if "walmart" in label_norm:
        return "walmart_prepaid_offers", "prepaid_retail"
    return "unknown", "unknown"


def parse_toc_page_map(pages_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER TOC PAGE MAP - QA ONLY")
    rows: List[Dict[str, Any]] = []
    toc_pages = pages_df[pages_df["raw_text"].fillna("").str.contains("Table of Contents", case=False, regex=False)]
    for _, toc_row in toc_pages.iterrows():
        source_page = int(toc_row["page_num"])
        for line in str(toc_row.get("raw_text") or "").splitlines():
            line_clean = clean_text_line(line)
            if not line_clean:
                continue
            tail = line_clean[-70:]
            page_nums = [int(x) for x in re.findall(r"\b\d{1,2}\b", tail) if int(x) >= 4]
            if not page_nums:
                continue
            label = re.sub(r"[\.…。]+", " ", line_clean)
            label = re.sub(r"^[▪❖•\-\s]+", "", label).strip()
            label = re.sub(r"(?:\s+\d{1,2}\s*(?:and|&|,)?\s*)+", "", label, flags=re.IGNORECASE).strip()
            if len(label) < 4:
                continue
            for expected_page_num in page_nums:
                expected_section_id, expected_group = infer_expected_section_from_toc_label(label, expected_page_num, page_nums)
                rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "toc_source_page_num": source_page, "toc_line_text": line_clean, "toc_label": label, "expected_page_num": expected_page_num, "expected_section_id": expected_section_id, "expected_section_group": expected_group, "created_ts": now_utc()})
    df = pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame()
    print(f"TOC map rows: {len(df):,}")
    return df


def infer_business_subsection(section_id: str, raw_text: str) -> str:
    if section_id != "business_device_offers":
        return section_id
    norm = normalize_for_match(raw_text)
    has_device = bool(re.search(r"\biphone\b|\bgalaxy\b|\bpixel\b|\bsamsung\b", norm))
    has_newline_byod = bool(re.search(r"\bnew line offers\b|\bbyod offers\b", norm))
    if has_newline_byod and not has_device:
        return "business_newline_byod_offers"
    return section_id


def allowed_carriers_for_section(section_id: str) -> List[str]:
    group = SECTION_GROUP_MAP.get(section_id, "")
    if group in {"postpaid", "postpaid_bts", "business"}:
        return ["T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Optimum"]
    if group == "cable_mvno":
        return ["Xfinity", "Spectrum", "Optimum"]
    if group in {"prepaid", "prepaid_price_grid", "prepaid_retail"}:
        return ["Metro", "Boost", "Cricket", "T-Mobile", "AT&T", "Verizon", "Straight Talk", "Total Wireless", "Walmart"]
    return ALLOWED_CARRIERS_DEFAULT


def build_context_pack(page_num: int, section_id: str, layout_type: str, title_header_text: str, raw_text: str) -> Dict[str, Any]:
    return {
        "page_num": page_num,
        "section_id": section_id,
        "section_group": SECTION_GROUP_MAP.get(section_id, "other"),
        "layout_type": layout_type,
        "expected_table_type": EXPECTED_TABLE_TYPE_BY_SECTION.get(section_id),
        "allowed_carriers": allowed_carriers_for_section(section_id),
        "expected_output": "table_cells_first",
        "title_header_text": title_header_text,
        "do_not_extract": ["footer", "page number", "confidentiality label", "source note"],
        "raw_text_preview": raw_text[:2000],
    }


def classify_all_slides(pages_df: pd.DataFrame, lines_df: pd.DataFrame, toc_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER SLIDE CLASSIFICATION AND CONTEXT PACKS")
    print_active_classifier_rules()
    toc_expected_by_page: Dict[int, str] = {}
    if toc_df is not None and not toc_df.empty:
        for _, row in toc_df.iterrows():
            toc_expected_by_page[int(row["expected_page_num"])] = str(row["expected_section_id"])
    rows: List[Dict[str, Any]] = []
    for _, page_row in pages_df.sort_values("page_num").iterrows():
        page_num = int(page_row["page_num"])
        raw_text = page_row.get("raw_text") or ""
        title_lines, title_header_text, header_norm = extract_title_header_lines(page_num, pages_df, lines_df)
        match = priority_classify(page_num, raw_text, header_norm)
        method = "priority_rule"
        confidence = 0.99
        if match is None:
            match = match_rules(TITLE_FIRST_OVERRIDES, header_norm)
            method = "title_first_override"
            confidence = 0.98
        if match is None:
            match = match_rules(SLIDE_HEADER_RULES, header_norm)
            method = "header_registry_title_zone_only"
            confidence = 0.94
        if match is None:
            match = {"section_id": "unknown", "rule_name": "unknown", "matched_pattern": None}
            method = "unknown"
            confidence = 0.0
        section_id = infer_business_subsection(match["section_id"], raw_text)
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        section_group = SECTION_GROUP_MAP.get(section_id, "other")
        expected_table_type = EXPECTED_TABLE_TYPE_BY_SECTION.get(section_id)
        context_pack = build_context_pack(page_num, section_id, layout_type, title_header_text, raw_text)
        toc_expected = toc_expected_by_page.get(page_num)
        rows.append({
            "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week,
            "page_num": page_num, "section_id": section_id, "section_group": section_group, "layout_type": layout_type,
            "requires_vision_extraction": bool(layout_type in VISION_LAYOUT_TYPES), "expected_table_type": expected_table_type,
            "extraction_route": "gemini_table_cells" if layout_type in VISION_LAYOUT_TYPES else "deterministic_text",
            "classifier_method": method, "classifier_confidence": confidence, "matched_rule": match.get("rule_name"),
            "matched_pattern": match.get("matched_pattern"), "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines), "toc_expected_section": toc_expected,
            "toc_mismatch_flag": bool(toc_expected and toc_expected != section_id),
            "classification_debug_json": safe_json({"header_norm": header_norm, "title_lines": title_lines}),
            "context_pack_json": safe_json(context_pack), "created_ts": now_utc(),
        })
    df = pd.DataFrame(rows)
    print(f"Slide section rows: {len(df):,}")
    if PRINT_CLASSIFIER_QA:
        print_subheader("Classifier QA by page")
        show_df(df[["page_num", "section_id", "section_group", "layout_type", "classifier_method", "toc_expected_section", "toc_mismatch_flag", "title_header_text"]], max_rows=60)
    if PRINT_CONTEXT_PACK_EXAMPLES:
        print_subheader("Example context packs for agent")
        sample = df[df["requires_vision_extraction"] == True].head(3)
        for _, row in sample.iterrows():
            print(f"Page {row['page_num']} -> {row['section_id']}")
            print(row["context_pack_json"][:1200])
    return df

# ======================================================================================
# SECTION 06 - BRONZE REGIONS, DETECTED TABLES, CONTENT BLOCKS
# ======================================================================================

def build_bronze_regions_and_tables(pages_df: pd.DataFrame, sections_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("BRONZE REGIONS AND DETECTED TABLE PLACEHOLDERS")
    region_rows: List[Dict[str, Any]] = []
    table_rows: List[Dict[str, Any]] = []
    for _, sec in sections_df.iterrows():
        page_num = int(sec["page_num"])
        page_row = pages_df[pages_df["page_num"] == page_num].iloc[0]
        width = float(page_row.get("page_width") or 0)
        height = float(page_row.get("page_height") or 0)
        section_id = sec["section_id"]
        layout_type = sec["layout_type"]
        raw_text = page_row.get("raw_text") or ""
        title = sec.get("title_header_text") or ""
        # Every slide gets a title region and a body/table region for QA.
        region_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "page_num": page_num, "region_id": make_hash_id("region", deck_id, page_num, "title"), "section_id": section_id, "region_type": "title", "region_label": "title_header", "x0": 0.0, "y0": 0.0, "x1": width, "y1": height * 0.18 if height else None, "region_text": title, "detected_by": "layout_rule", "confidence": float(sec.get("classifier_confidence") or 0), "created_ts": now_utc()})
        body_region_type = "table_body" if layout_type in VISION_LAYOUT_TYPES else "text_body"
        region_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "page_num": page_num, "region_id": make_hash_id("region", deck_id, page_num, body_region_type), "section_id": section_id, "region_type": body_region_type, "region_label": layout_type, "x0": 0.0, "y0": height * 0.18 if height else None, "x1": width, "y1": height, "region_text": raw_text[:5000], "detected_by": "layout_rule", "confidence": 0.75, "created_ts": now_utc()})
        table_type = sec.get("expected_table_type")
        if table_type:
            table_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "page_num": page_num, "table_id": make_hash_id("table", deck_id, page_num, table_type), "section_id": section_id, "layout_type": layout_type, "table_type": table_type, "table_title": title, "x0": 0.0, "y0": height * 0.18 if height else None, "x1": width, "y1": height, "detected_by": "section_classifier", "confidence": float(sec.get("classifier_confidence") or 0), "raw_table_text": raw_text[:12000], "created_ts": now_utc()})
    regions_df = pd.DataFrame(region_rows)
    tables_df = pd.DataFrame(table_rows)
    print(f"bronze_slideRegions rows: {len(regions_df):,}")
    print(f"bronze_slideDetectedTables rows: {len(tables_df):,}")
    return regions_df, tables_df


def infer_carrier_from_text(text: str) -> Optional[str]:
    if not text:
        return None
    norm = normalize_for_match(text)
    patterns = [("T-Mobile", r"\bt mobile\b|\btmobile\b"), ("AT&T", r"\bat and t\b|\batt\b"), ("Verizon", r"\bverizon\b|\bvzw\b"), ("Xfinity", r"\bxfinity\b|\bcomcast\b"), ("Spectrum", r"\bspectrum\b"), ("Metro", r"\bmetro\b"), ("Boost", r"\bboost\b"), ("Cricket", r"\bcricket\b"), ("Total Wireless", r"\btotal wireless\b"), ("Straight Talk", r"\bstraight talk\b"), ("Walmart", r"\bwalmart\b")]
    for carrier, pattern in patterns:
        if re.search(pattern, norm):
            return carrier
    return None


def make_content_block(deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date, page_num: int, section_id: str, section_group: str, layout_type: str, block_type: str, carrier: Optional[str], brand: Optional[str], category: Optional[str], block_title: Optional[str], block_text: str, source_method: str, confidence: float, needs_review: bool) -> Dict[str, Any]:
    block_id = make_hash_id("block", deck_id, page_num, section_id, block_type, carrier, block_title, block_text)
    return {"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "block_id": block_id, "page_num": page_num, "section_id": section_id, "section_group": section_group, "layout_type": layout_type, "block_type": block_type, "carrier": carrier, "brand": brand, "category": category, "block_title": block_title, "block_text": block_text, "source_method": source_method, "confidence": confidence, "needs_review": needs_review, "created_ts": now_utc()}


def build_ci_headline_blocks(raw_text: str, base: Dict[str, Any]) -> List[Dict[str, Any]]:
    blocks: List[Dict[str, Any]] = []
    lines = [clean_text_line(x) for x in raw_text.splitlines() if clean_text_line(x)]
    current_carrier: Optional[str] = None
    current_lines: List[str] = []
    rank = 0
    def flush() -> None:
        nonlocal rank, current_carrier, current_lines
        if current_carrier and current_lines:
            rank += 1
            text = "\n".join(current_lines)
            blocks.append(make_content_block(**base, block_type="ci_headline", carrier=current_carrier, brand=None, category=str(rank), block_title=current_carrier, block_text=text, source_method="deterministic_numbered_headlines", confidence=0.90, needs_review=False))
    for line in lines:
        match = re.match(r"^\s*(\d{1,2})\.\s+(.+?)\s*$", line)
        if match:
            flush()
            current_carrier = clean_text_line(match.group(2))
            current_lines = []
        elif current_carrier:
            current_lines.append(line)
    flush()
    return blocks


def build_major_change_blocks(raw_text: str, base: Dict[str, Any]) -> List[Dict[str, Any]]:
    blocks: List[Dict[str, Any]] = []
    current_carrier: Optional[str] = None
    current_change_type = "unknown_change"
    for raw_line in raw_text.splitlines():
        line = clean_text_line(raw_line)
        if not line:
            continue
        norm = normalize_for_match(line)
        if norm in {"offers introduced", "introduced"}:
            current_change_type = "introduced"
            continue
        if norm in {"offers removed", "removed"}:
            current_change_type = "removed"
            continue
        carrier = infer_carrier_from_text(line)
        if carrier and len(line) <= 40:
            current_carrier = carrier
            continue
        if re.match(r"^[▪•o\-]\s*", line) or re.search(r"\$\d+|\bfree\b|\bno changes\b|\bno offers\b|\bremoved\b|\bexpired\b", line, flags=re.IGNORECASE):
            text = re.sub(r"^[▪•o\-]\s*", "", line).strip()
            if len(text) >= 2:
                blocks.append(make_content_block(**base, block_type="major_offer_change", carrier=current_carrier, brand=None, category=current_change_type, block_title=current_change_type, block_text=text, source_method="deterministic_major_change_lines", confidence=0.72, needs_review=True))
    return blocks


def build_content_blocks(pages_df: pd.DataFrame, sections_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER CONTENT BLOCK CREATION")
    rows: List[Dict[str, Any]] = []
    for _, sec in sections_df.iterrows():
        page_num = int(sec["page_num"])
        raw_text = pages_df[pages_df["page_num"] == page_num].iloc[0].get("raw_text") or ""
        section_id = sec["section_id"]
        layout_type = sec["layout_type"]
        section_group = sec["section_group"]
        base = {"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "page_num": page_num, "section_id": section_id, "section_group": section_group, "layout_type": layout_type}
        if section_id in {"cover", "confidentiality_notice", "table_of_contents", "section_divider_ci_headlines", "section_divider_offer_updates", "section_divider_postpaid", "section_divider_prepaid", "end_slide"}:
            rows.append(make_content_block(**base, block_type="non_offer_slide", carrier=None, brand=None, category=None, block_title=section_id, block_text=raw_text[:3000], source_method="non_offer_capture", confidence=0.99, needs_review=False))
        elif section_id in {"ci_headlines_postpaid", "ci_headlines_prepaid"}:
            rows.extend(build_ci_headline_blocks(raw_text, base))
        elif section_id == "major_offer_changes":
            rows.extend(build_major_change_blocks(raw_text, base))
        elif section_id == "national_retail_readout":
            for line in [clean_text_line(x) for x in raw_text.splitlines() if clean_text_line(x)]:
                if re.search(r"\$\d+|\bpromo|promotion|best buy|walmart|target|costco|device", line, flags=re.IGNORECASE):
                    rows.append(make_content_block(**base, block_type="retail_readout_line", carrier=infer_carrier_from_text(line), brand=None, category="retail_readout", block_title="retail_readout", block_text=line, source_method="deterministic_retail_lines", confidence=0.55, needs_review=True))
        elif layout_type in VISION_LAYOUT_TYPES:
            rows.append(make_content_block(**base, block_type="vision_table_placeholder", carrier=None, brand=None, category=sec.get("expected_table_type"), block_title=section_id, block_text=raw_text[:5000], source_method="vision_table_cell_routing", confidence=0.80, needs_review=not USE_GEMINI_VISION))
        else:
            rows.append(make_content_block(**base, block_type="unknown_page", carrier=None, brand=None, category=None, block_title=section_id, block_text=raw_text[:5000], source_method="unknown_capture", confidence=0.0, needs_review=True))
    df = pd.DataFrame(rows)
    print(f"silver_slideContentBlocks rows: {len(df):,}")
    return df

# ======================================================================================
# SECTION 07 - GEMINI VISION AS TABLE-CELL RECONSTRUCTION AGENT
# ======================================================================================

def make_gemini_client():
    if not GOOGLE_GENAI_AVAILABLE:
        raise RuntimeError("google-genai is not available. Install/import google-genai first.")
    return genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)


def parse_gemini_json_response(text: str) -> Dict[str, Any]:
    if not text:
        raise ValueError("Empty Gemini response.")
    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end + 1])
    raise ValueError("Could not parse Gemini response as valid JSON.")


def build_gemini_cell_prompt(context_pack: Dict[str, Any], raw_text: str) -> str:
    section_id = context_pack.get("section_id")
    layout_type = context_pack.get("layout_type")
    table_type = context_pack.get("expected_table_type")
    allowed_carriers = context_pack.get("allowed_carriers", [])
    title_header_text = context_pack.get("title_header_text", "")
    prompt = f"""
You are extracting the visual table structure from one telecom competitive-offer slide.

Your task is NOT to create final offer rows. Your task is to reconstruct table cells and preserve context.
Use the rendered slide image as the primary source. Use raw PDF text only as a secondary aid.

Slide context:
section_id: {section_id}
layout_type: {layout_type}
expected_table_type: {table_type}
title_header_text: {title_header_text}
allowed_carriers: {json.dumps(allowed_carriers)}

Rules:
1. Return valid JSON only. No markdown.
2. Extract table cells or meaningful row/cell-like records.
3. Preserve row_header, column_header, super_column_header, and carrier_context when visible.
4. For non-price-grid tables, use cell_text to capture the offer statement in the correct carrier column.
5. For price-grid pages, return one cell per device/scenario/price where possible.
6. Do not extract footer, page number, confidentiality label, or source note.
7. Do not invent carriers or devices.
8. Use needs_review=true when visual alignment is uncertain.

Return exactly this JSON shape:
{{
  "extraction_summary": {{
    "section_id": "{section_id}",
    "layout_type": "{layout_type}",
    "table_type": "{table_type}",
    "confidence": 0.0,
    "notes": null
  }},
  "tables": [
    {{
      "table_title": null,
      "table_type": "{table_type}",
      "confidence": 0.0
    }}
  ],
  "cells": [
    {{
      "row_index": null,
      "column_index": null,
      "row_header": null,
      "column_header": null,
      "super_column_header": null,
      "cell_text": null,
      "carrier_context": null,
      "brand_context": null,
      "product_context": null,
      "category_context": null,
      "raw_evidence": null,
      "confidence": 0.0,
      "needs_review": true
    }}
  ]
}}

Raw PDF text for reference:
{raw_text[:9000]}
"""
    return textwrap.dedent(prompt).strip()


def sanitize_agent_cells(parsed_json: Dict[str, Any], section_id: str, layout_type: str) -> Dict[str, Any]:
    if not isinstance(parsed_json, dict):
        return {"extraction_summary": {}, "tables": [], "cells": []}
    cells = parsed_json.get("cells", []) or []
    clean_cells = []
    allowed_carriers = set(allowed_carriers_for_section(section_id))
    for cell in cells:
        if not isinstance(cell, dict):
            continue
        cell_text = clean_text_line(cell.get("cell_text"))
        evidence = clean_text_line(cell.get("raw_evidence") or cell_text)
        if not cell_text and not evidence:
            continue
        carrier = cell.get("carrier_context")
        if carrier and carrier not in allowed_carriers:
            cell["needs_review"] = True
        cell["cell_text"] = cell_text or evidence
        cell["raw_evidence"] = evidence or cell_text
        clean_cells.append(cell)
    parsed_json["cells"] = clean_cells
    return parsed_json


def call_gemini_for_slide_cells(page_row: pd.Series, section_row: pd.Series) -> Dict[str, Any]:
    client = make_gemini_client()
    raw_text = page_row.get("raw_text") or ""
    image_path = str(page_row.get("rendered_image_path"))
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    context_pack = json.loads(section_row.get("context_pack_json") or "{}")
    prompt = build_gemini_cell_prompt(context_pack, raw_text)
    image_part = genai_types.Part.from_bytes(data=image_bytes, mime_type="image/png")
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[prompt, image_part],
        config=genai_types.GenerateContentConfig(temperature=0.0, response_mime_type="application/json"),
    )
    response_text = getattr(response, "text", None) or getattr(response, "output_text", None)
    parsed_json = parse_gemini_json_response(response_text)
    parsed_json = sanitize_agent_cells(parsed_json, str(section_row["section_id"]), str(section_row["layout_type"]))
    return {"parsed_json": parsed_json, "raw_response_text": response_text, "prompt_text": prompt, "context_pack": context_pack}


def run_gemini_vision_cell_extraction(pages_df: pd.DataFrame, sections_df: pd.DataFrame, tables_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("GEMINI VISION EXTRACTION - TABLE CELLS FIRST")
    raw_rows: List[Dict[str, Any]] = []
    cell_rows: List[Dict[str, Any]] = []
    if not USE_GEMINI_VISION:
        print("USE_GEMINI_VISION is False. Skipping agent extraction.")
        return pd.DataFrame(raw_rows), pd.DataFrame(cell_rows)
    queued = sections_df[(sections_df["requires_vision_extraction"] == True) & (sections_df["section_id"] != "unknown")].copy()
    print(f"Gemini table-cell pages queued: {len(queued):,}")
    processed = 0
    for _, sec in queued.sort_values("page_num").iterrows():
        if processed >= MAX_GEMINI_PAGES_PER_RUN:
            break
        page_num = int(sec["page_num"])
        page_row = pages_df[pages_df["page_num"] == page_num].iloc[0]
        table_id = make_hash_id("table", deck_id, page_num, sec.get("expected_table_type"))
        print_subheader(f"Gemini table-cell page {page_num}: {sec['section_id']} | {sec['layout_type']}")
        parsed_json = None
        raw_response_text = None
        prompt_text = None
        context_pack = json.loads(sec.get("context_pack_json") or "{}")
        status = "success"
        error_message = None
        try:
            result = call_gemini_for_slide_cells(page_row, sec)
            parsed_json = result["parsed_json"]
            raw_response_text = result["raw_response_text"]
            prompt_text = result["prompt_text"]
            context_pack = result["context_pack"]
            cells = parsed_json.get("cells", []) or []
            print(f"Agent cells returned: {len(cells):,}")
            for i, cell in enumerate(cells):
                cell_id = make_hash_id("cell", deck_id, page_num, sec["section_id"], cell.get("row_header"), cell.get("column_header"), cell.get("cell_text"), i)
                confidence = safe_float(cell.get("confidence")) if cell.get("confidence") is not None else 0.75
                cell_rows.append({
                    "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week,
                    "page_num": page_num, "table_id": table_id, "cell_id": cell_id, "section_id": sec["section_id"],
                    "layout_type": sec["layout_type"], "table_type": sec.get("expected_table_type"),
                    "row_index": safe_float(cell.get("row_index")), "column_index": safe_float(cell.get("column_index")),
                    "row_header": cell.get("row_header"), "column_header": cell.get("column_header"),
                    "super_column_header": cell.get("super_column_header"), "cell_text": cell.get("cell_text"),
                    "cell_text_norm": normalize_for_match(cell.get("cell_text")), "carrier_context": cell.get("carrier_context"),
                    "brand_context": cell.get("brand_context"), "product_context": cell.get("product_context"),
                    "category_context": cell.get("category_context"), "x0": None, "y0": None, "x1": None, "y1": None,
                    "detected_by": "gemini_vision_table_cells", "confidence": confidence,
                    "needs_review": bool(cell.get("needs_review", False)) or confidence < AUTO_APPROVE_MIN_CONFIDENCE,
                    "raw_evidence": cell.get("raw_evidence") or cell.get("cell_text"), "created_ts": now_utc(),
                })
        except Exception as exc:
            status = "error"
            error_message = traceback.format_exc()
            print(f"Gemini failed on page {page_num}: {exc}")
            print(error_message[:2000])
        raw_rows.append({
            "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week,
            "page_num": page_num, "section_id": sec["section_id"], "layout_type": sec["layout_type"],
            "agent_task": "table_cell_reconstruction", "extractor_name": "gemini_vision_table_cell_agent",
            "extractor_model": GEMINI_MODEL, "prompt_version": PROMPT_VERSION,
            "context_pack_json": safe_json(context_pack), "prompt_text": prompt_text,
            "raw_json": safe_json(parsed_json), "raw_response_text": raw_response_text,
            "extraction_status": status, "error_message": error_message, "created_ts": now_utc(),
        })
        processed += 1
        time.sleep(GEMINI_SLEEP_SECONDS)
    raw_df = pd.DataFrame(raw_rows)
    cells_df = pd.DataFrame(cell_rows)
    print(f"bronze_agentExtractionsRaw rows: {len(raw_df):,}")
    print(f"bronze_slideTableCells rows: {len(cells_df):,}")
    return raw_df, cells_df

# ======================================================================================
# SECTION 08 - VALUE MENTIONS AND SILVER OFFER NORMALIZATION
# ======================================================================================

def extract_value_mentions_from_text(source_text: str, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date, page_num: int, section_id: str, source_table: str, source_id: str) -> List[Dict[str, Any]]:
    text = source_text or ""
    rows: List[Dict[str, Any]] = []
    patterns = [
        ("free", r"\bFREE\b|\bFree\b|\bOn Us\b|\bon us\b"),
        ("bogo", r"\bBOGO\b|\bbuy\s+one\b"),
        ("currency_amount", r"(?:up to\s*)?\$\d[\d,]*(?:\.\d{1,2})?"),
        ("currency_range", r"\$\d[\d,]*(?:\.\d{1,2})?\s*/\s*\$?\d[\d,]*(?:\.\d{1,2})?"),
        ("percentage", r"\d+(?:\.\d+)?%"),
        ("data_allowance_gb", r"\d+(?:\.\d+)?\s*GB\b"),
        ("term_months", r"\d+\s*[- ]?mo\.?|\d+\s*months?"),
        ("date_range", r"\d{1,2}/\d{1,2}(?:/\d{2,4})?\s*[–\-]\s*(?:\d{1,2}/\d{1,2}(?:/\d{2,4})?|TBD)?"),
        ("no_offer", r"\bNo offer\b|\bNo changes\b|\bNo new offers\b|\bNo expired offers\b|\bNo offers expired\b"),
    ]
    for value_type, pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            value_text = match.group(0)
            start, end = match.span()
            before = text[max(0, start - 80):start]
            after = text[end:end + 80]
            value_num = safe_float(value_text)
            value_num_min = None
            value_num_max = None
            comparator = None
            if "up to" in value_text.lower():
                comparator = "up_to"
                value_num_max = value_num
                value_num = None
            if value_type == "currency_range":
                nums = [safe_float(x) for x in re.findall(r"\$?\d[\d,]*(?:\.\d{1,2})?", value_text)]
                nums = [x for x in nums if x is not None]
                if nums:
                    value_num_min = min(nums)
                    value_num_max = max(nums)
                    value_num = None
                    comparator = "range"
            rows.append({
                "deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week,
                "page_num": page_num, "source_table": source_table, "source_id": source_id, "section_id": section_id,
                "value_mention_id": make_hash_id("vm", source_id, value_type, value_text, start),
                "source_text": text[:2000], "value_text": value_text, "value_type": value_type,
                "value_num": value_num, "value_num_min": value_num_min, "value_num_max": value_num_max,
                "currency": "USD" if "$" in value_text or value_type in {"free", "currency_amount", "currency_range"} else None,
                "unit": "GB" if value_type == "data_allowance_gb" else None,
                "period": "months" if value_type == "term_months" else None,
                "comparator": comparator, "context_before": before, "context_after": after,
                "confidence": 0.90, "created_ts": now_utc(),
            })
    return rows


def build_bronze_value_mentions(blocks_df: pd.DataFrame, cells_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("BRONZE VALUE MENTION EXTRACTION")
    rows: List[Dict[str, Any]] = []
    if blocks_df is not None and not blocks_df.empty:
        for _, b in blocks_df.iterrows():
            rows.extend(extract_value_mentions_from_text(str(b.get("block_text") or ""), deck_id, run_id, pdf_hash, pdf_name, deck_week, int(b["page_num"]), str(b["section_id"]), "silver_slideContentBlocks", str(b["block_id"])))
    if cells_df is not None and not cells_df.empty:
        for _, c in cells_df.iterrows():
            rows.extend(extract_value_mentions_from_text(str(c.get("cell_text") or ""), deck_id, run_id, pdf_hash, pdf_name, deck_week, int(c["page_num"]), str(c["section_id"]), "bronze_slideTableCells", str(c["cell_id"])))
    df = pd.DataFrame(rows).drop_duplicates(subset=["value_mention_id"]) if rows else pd.DataFrame()
    print(f"bronze_valueMentions rows: {len(df):,}")
    return df


def infer_offer_type(text: str) -> str:
    norm = normalize_for_match(text)
    if re.search(r"\bno offer\b|\bno changes\b|\bno new offers\b|\bno expired offers\b", norm):
        return "no_offer"
    if "bogo" in norm:
        return "bogo"
    if "trade" in norm:
        return "trade_in_discount"
    if "port" in norm:
        return "port_in_discount"
    if "byod" in norm or "bring your own" in norm:
        return "byod_credit"
    if "bill credit" in norm or "credit" in norm:
        return "bill_credit"
    if "free" in norm or "on us" in norm:
        return "free_or_on_us"
    if re.search(r"\$\d", text or ""):
        return "discount_or_price"
    if "removed" in norm or "expired" in norm or "retired" in norm:
        return "expired_or_removed"
    return "other"


def extract_product_name(text: str, fallback: Optional[str] = None) -> Optional[str]:
    if not text:
        return fallback
    patterns = [
        r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+e)?",
        r"Galaxy\s+[A-Z]?\d+[A-Za-z]*(?:\s+5G|\s+FE|\s+Ultra)?",
        r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?",
        r"moto\s+g\s+(?:stylus\s+)?(?:power\s+)?5G\s+\d{4}",
        r"moto\s+razr\s+\d{4}", r"Apple\s+Watch\s+[A-Za-z0-9\s]+", r"iPad\s+[A-Za-z0-9\s]+",
        r"TCL\s+[A-Za-z0-9\s]+", r"REVVL\s+[A-Za-z0-9\s]+", r"GizmoWatch\s+\d*",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return clean_text_line(match.group(0))
    return fallback


def infer_product_category(product_name: Optional[str], section_id: str, text: str) -> Optional[str]:
    norm = normalize_for_match(" ".join([product_name or "", section_id or "", text or ""]))
    if "watch" in norm:
        return "watch"
    if "tablet" in norm or "ipad" in norm or "tab " in norm:
        return "tablet"
    if "gateway" in norm or "hint" in norm or "internet" in norm:
        return "home_internet"
    if "byod" in norm:
        return "byod"
    if any(x in norm for x in ["iphone", "galaxy", "pixel", "moto", "phone"]):
        return "smartphone"
    if "plan" in norm or "service" in norm:
        return "service_plan"
    return None


def build_candidates_from_blocks(blocks_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    if blocks_df is None or blocks_df.empty:
        return pd.DataFrame()
    for _, b in blocks_df.iterrows():
        if b.get("block_type") in {"non_offer_slide", "vision_table_placeholder", "unknown_page", "ci_headline"}:
            continue
        text = str(b.get("block_text") or "")
        offer_type = infer_offer_type(text)
        product = extract_product_name(text)
        value = safe_float(text)
        candidate_id = make_hash_id("cand", b.get("block_id"), text, b.get("carrier"))
        confidence = safe_float(b.get("confidence")) or 0.55
        errors = []
        if not b.get("carrier") and b.get("section_id") != "national_retail_readout":
            errors.append("missing_carrier")
        rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "candidate_id": candidate_id, "block_id": b.get("block_id"), "cell_id": None, "page_num": int(b.get("page_num")), "section_id": b.get("section_id"), "section_group": b.get("section_group"), "layout_type": b.get("layout_type"), "carrier": b.get("carrier"), "brand": b.get("brand"), "product_name": product, "product_category": infer_product_category(product, str(b.get("section_id")), text), "offer_type": offer_type, "offer_summary": text[:1000], "offer_amount_text": text if re.search(r"\$\d|free|on us|bogo", text, flags=re.IGNORECASE) else None, "offer_value": value, "currency": "USD" if re.search(r"\$\d|free|on us", text, flags=re.IGNORECASE) else None, "rate_plan_requirement": None, "customer_segment": None, "action_required": None, "date_start_text": None, "date_end_text": None, "condition_text": text, "raw_evidence": text, "extraction_method": "deterministic_block_parser", "confidence": confidence, "needs_review": True, "validation_errors_json": safe_json(errors), "created_ts": now_utc()})
    return pd.DataFrame(rows)


def build_candidates_and_price_rows_from_cells(cells_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    candidate_rows: List[Dict[str, Any]] = []
    price_rows: List[Dict[str, Any]] = []
    if cells_df is None or cells_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    for _, cell in cells_df.iterrows():
        text = str(cell.get("cell_text") or "")
        section_id = str(cell.get("section_id") or "")
        layout_type = str(cell.get("layout_type") or "")
        table_type = str(cell.get("table_type") or "")
        page_num = int(cell.get("page_num"))
        product = cell.get("product_context") or cell.get("row_header") or extract_product_name(text)
        carrier = cell.get("carrier_context") or infer_carrier_from_text(str(cell.get("column_header") or "")) or infer_carrier_from_text(text)
        confidence = safe_float(cell.get("confidence")) or 0.75
        if layout_type == "price_grid" or table_type == "metro_price_grid":
            price_text = text if re.search(r"\$\d|FREE|Free|N/A", text) else None
            price_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "grid_row_id": make_hash_id("grid", cell.get("cell_id"), product, cell.get("column_header"), text), "cell_id": cell.get("cell_id"), "page_num": page_num, "section_id": section_id, "section_group": SECTION_GROUP_MAP.get(section_id), "brand": cell.get("brand_context") or "Metro", "carrier": carrier or "Metro", "device_name": product, "device_category": infer_product_category(str(product), section_id, text), "retail_price": None, "customer_scenario": cell.get("super_column_header"), "id_requirement": None, "rate_plan_requirement": cell.get("column_header"), "price_text": price_text, "price_value": safe_float(price_text), "savings_text": None, "savings_value": None, "raw_evidence": cell.get("raw_evidence") or text, "extraction_method": "gemini_cell_to_price_grid", "confidence": confidence, "needs_review": bool(cell.get("needs_review")), "validation_errors_json": safe_json([]), "created_ts": now_utc()})
        else:
            offer_type = infer_offer_type(text)
            candidate_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "candidate_id": make_hash_id("cand", cell.get("cell_id"), carrier, product, text), "block_id": None, "cell_id": cell.get("cell_id"), "page_num": page_num, "section_id": section_id, "section_group": SECTION_GROUP_MAP.get(section_id), "layout_type": layout_type, "carrier": carrier, "brand": cell.get("brand_context"), "product_name": product, "product_category": infer_product_category(str(product), section_id, text), "offer_type": offer_type, "offer_summary": text[:1000], "offer_amount_text": text if re.search(r"\$\d|free|on us|bogo", text, flags=re.IGNORECASE) else None, "offer_value": safe_float(text), "currency": "USD" if re.search(r"\$\d|free|on us", text, flags=re.IGNORECASE) else None, "rate_plan_requirement": cell.get("column_header"), "customer_segment": cell.get("super_column_header"), "action_required": None, "date_start_text": None, "date_end_text": None, "condition_text": text, "raw_evidence": cell.get("raw_evidence") or text, "extraction_method": "gemini_cell_parser", "confidence": confidence, "needs_review": bool(cell.get("needs_review")) or confidence < AUTO_APPROVE_MIN_CONFIDENCE, "validation_errors_json": safe_json([]), "created_ts": now_utc()})
    return pd.DataFrame(candidate_rows), pd.DataFrame(price_rows)


def build_offer_candidates(blocks_df: pd.DataFrame, cells_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("SILVER OFFER CANDIDATES AND PRICE GRID ROWS")
    block_candidates = build_candidates_from_blocks(blocks_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    cell_candidates, price_rows = build_candidates_and_price_rows_from_cells(cells_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    frames = [df for df in [block_candidates, cell_candidates] if df is not None and not df.empty]
    candidates = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["candidate_id"]) if frames else pd.DataFrame()
    print(f"silver_offerCandidates rows: {len(candidates):,}")
    print(f"silver_priceGridRows rows: {len(price_rows):,}")
    return candidates, price_rows


def build_silver_offer_values(candidates_df: pd.DataFrame, value_mentions_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER OFFER VALUES")
    rows: List[Dict[str, Any]] = []
    if candidates_df is None or candidates_df.empty:
        return pd.DataFrame()
    for _, c in candidates_df.iterrows():
        text = str(c.get("raw_evidence") or "")
        mentions = extract_value_mentions_from_text(text, deck_id, run_id, pdf_hash, pdf_name, deck_week, int(c["page_num"]), str(c["section_id"]), "silver_offerCandidates", str(c["candidate_id"]))
        for i, m in enumerate(mentions):
            role = "primary" if i == 0 else "secondary"
            rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "offer_value_id": make_hash_id("ov", c["candidate_id"], m["value_text"], i), "candidate_id": c["candidate_id"], "page_num": int(c["page_num"]), "section_id": c["section_id"], "value_role": role, "value_type": m["value_type"], "value_text": m["value_text"], "value_num": m["value_num"], "value_num_min": m["value_num_min"], "value_num_max": m["value_num_max"], "currency": m["currency"], "unit": m["unit"], "period": m["period"], "comparator": m["comparator"], "raw_evidence": text, "confidence": m["confidence"], "created_ts": now_utc()})
    df = pd.DataFrame(rows).drop_duplicates(subset=["offer_value_id"]) if rows else pd.DataFrame()
    print(f"silver_offerValues rows: {len(df):,}")
    return df

# ======================================================================================
# SECTION 09 - NORMALIZED OFFERS, DEVICE BRIDGE, REVIEW, AND GOLD TABLES
# ======================================================================================

def infer_device_brand(device_name: Optional[str]) -> Optional[str]:
    norm = normalize_for_match(device_name)
    if not norm:
        return None
    if "iphone" in norm or "ipad" in norm or "apple watch" in norm:
        return "Apple"
    if "galaxy" in norm or "samsung" in norm:
        return "Samsung"
    if "pixel" in norm:
        return "Google"
    if "moto" in norm or "razr" in norm or "motorola" in norm:
        return "Motorola"
    if "tcl" in norm:
        return "TCL"
    if "revvl" in norm:
        return "T-Mobile"
    return None


def normalize_offers(candidates_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER NORMALIZED OFFERS")
    rows: List[Dict[str, Any]] = []
    if candidates_df is None or candidates_df.empty:
        return pd.DataFrame()
    for _, c in candidates_df.iterrows():
        confidence = safe_float(c.get("confidence")) or 0.0
        method = str(c.get("extraction_method") or "")
        is_gemini = method.startswith("gemini")
        needs_review = bool(c.get("needs_review")) or confidence < AUTO_APPROVE_MIN_CONFIDENCE
        auto_flag = bool((not needs_review) and c.get("offer_type") != "no_offer" and ((is_gemini and AUTO_APPROVE_GEMINI_ROWS) or ((not is_gemini) and AUTO_APPROVE_DETERMINISTIC_ROWS)))
        curation_status = "auto_approved" if auto_flag else "pending_review"
        offer_id = make_hash_id("offer", c.get("candidate_id"), c.get("carrier"), c.get("product_name"), c.get("raw_evidence"))
        rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "offer_id": offer_id, "candidate_id": c.get("candidate_id"), "page_num": int(c.get("page_num")), "section_id": c.get("section_id"), "section_group": c.get("section_group"), "layout_type": c.get("layout_type"), "carrier": c.get("carrier"), "brand": c.get("brand") or infer_device_brand(c.get("product_name")), "product_name": c.get("product_name"), "product_category": c.get("product_category") or infer_product_category(c.get("product_name"), str(c.get("section_id")), str(c.get("raw_evidence"))), "offer_type": c.get("offer_type"), "offer_summary": c.get("offer_summary"), "primary_value_text": c.get("offer_amount_text"), "primary_value_num": safe_float(c.get("offer_value")), "currency": c.get("currency"), "rate_plan_requirement": c.get("rate_plan_requirement"), "customer_segment": c.get("customer_segment"), "action_required": c.get("action_required"), "condition_text": c.get("condition_text"), "date_start": parse_date_safe(c.get("date_start_text")), "date_end": parse_date_safe(c.get("date_end_text")), "raw_evidence": c.get("raw_evidence"), "normalization_method": "v3_rule_normalizer", "confidence": confidence, "needs_review": needs_review, "auto_approve_flag": auto_flag, "curation_status": curation_status, "created_ts": now_utc()})
    df = pd.DataFrame(rows).drop_duplicates(subset=["offer_id"])
    print(f"silver_normalizedOffers rows: {len(df):,}")
    print(f"Auto approved rows: {int(df['auto_approve_flag'].sum()) if not df.empty else 0:,}")
    return df


def extract_device_names(text: str) -> List[str]:
    if not text:
        return []
    patterns = [r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+e)?", r"Galaxy\s+[A-Z]?\d+[A-Za-z]*(?:\s+5G|\s+FE|\s+Ultra)?", r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?", r"moto\s+g\s+(?:stylus\s+)?(?:power\s+)?5G\s+\d{4}", r"moto\s+razr\s+\d{4}", r"Apple\s+Watch\s+[A-Za-z0-9\s]+", r"iPad\s+[A-Za-z0-9\s]+", r"TCL\s+[A-Za-z0-9\s]+", r"REVVL\s+[A-Za-z0-9\s]+"]
    out: List[str] = []
    seen = set()
    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            value = clean_text_line(match.group(0))
            key = normalize_for_match(value)
            if value and key not in seen:
                out.append(value); seen.add(key)
    return out


def build_offer_device_bridge(normalized_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> pd.DataFrame:
    print_header("SILVER OFFER DEVICE BRIDGE")
    rows: List[Dict[str, Any]] = []
    if normalized_df is None or normalized_df.empty:
        return pd.DataFrame()
    for _, o in normalized_df.iterrows():
        devices = []
        if o.get("product_name"):
            devices.append(str(o.get("product_name")))
        devices.extend(extract_device_names(str(o.get("raw_evidence") or "")))
        seen = set()
        for d in devices:
            key = normalize_for_match(d)
            if not key or key in seen:
                continue
            seen.add(key)
            rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "bridge_id": make_hash_id("bridge", o.get("offer_id"), d), "offer_id": o.get("offer_id"), "candidate_id": o.get("candidate_id"), "page_num": int(o.get("page_num")), "section_id": o.get("section_id"), "carrier": o.get("carrier"), "device_name": d, "device_brand": infer_device_brand(d), "device_family": infer_product_category(d, str(o.get("section_id")), str(o.get("raw_evidence"))), "device_model": d, "raw_evidence": o.get("raw_evidence"), "confidence": o.get("confidence"), "created_ts": now_utc()})
    df = pd.DataFrame(rows).drop_duplicates(subset=["bridge_id"]) if rows else pd.DataFrame()
    print(f"silver_offerDeviceBridge rows: {len(df):,}")
    return df


def build_review_tables(candidates_df: pd.DataFrame, price_rows_df: pd.DataFrame, normalized_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date) -> Tuple[pd.DataFrame, pd.DataFrame]:
    print_header("REVIEW QUEUE AND DECISIONS")
    review_rows: List[Dict[str, Any]] = []
    decision_rows: List[Dict[str, Any]] = []
    if candidates_df is not None and not candidates_df.empty:
        for _, c in candidates_df.iterrows():
            if bool(c.get("needs_review")):
                review_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "review_id": make_hash_id("review", c.get("candidate_id")), "candidate_id": c.get("candidate_id"), "grid_row_id": None, "page_num": int(c.get("page_num")), "section_id": c.get("section_id"), "review_reason": c.get("validation_errors_json") or "needs_review", "review_status": "pending", "source_table": "silver_offerCandidates", "raw_payload_json": safe_json(c.to_dict()), "created_ts": now_utc()})
    if price_rows_df is not None and not price_rows_df.empty:
        for _, g in price_rows_df.iterrows():
            if bool(g.get("needs_review")):
                review_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "review_id": make_hash_id("review", g.get("grid_row_id")), "candidate_id": None, "grid_row_id": g.get("grid_row_id"), "page_num": int(g.get("page_num")), "section_id": g.get("section_id"), "review_reason": g.get("validation_errors_json") or "needs_review", "review_status": "pending", "source_table": "silver_priceGridRows", "raw_payload_json": safe_json(g.to_dict()), "created_ts": now_utc()})
    if normalized_df is not None and not normalized_df.empty:
        for _, o in normalized_df[normalized_df["auto_approve_flag"] == True].iterrows():
            decision_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "decision_id": make_hash_id("decision", o.get("candidate_id")), "candidate_id": o.get("candidate_id"), "grid_row_id": None, "page_num": int(o.get("page_num")), "section_id": o.get("section_id"), "decision_type": "auto_approved", "decision_status": "approved", "decision_source": "system_rules_v3", "decision_notes": f"confidence={o.get('confidence')}", "created_ts": now_utc()})
    review_df = pd.DataFrame(review_rows)
    decisions_df = pd.DataFrame(decision_rows)
    print(f"review_extractionReview rows: {len(review_df):,}")
    print(f"review_reviewDecisions rows: {len(decisions_df):,}")
    return review_df, decisions_df


def build_gold_tables(pdf_decks_df: pd.DataFrame, blocks_df: pd.DataFrame, normalized_df: pd.DataFrame, offer_values_df: pd.DataFrame, bridge_df: pd.DataFrame, price_rows_df: pd.DataFrame, deck_id: str, run_id: str, pdf_hash: str, pdf_name: str, deck_week: date, page_count: int) -> Dict[str, pd.DataFrame]:
    print_header("GOLD TABLE BUILD - BUSINESS CONSUMPTION TABLES")
    if normalized_df is None:
        normalized_df = pd.DataFrame()
    included = normalized_df.copy()
    if not LOAD_PENDING_ROWS_TO_GOLD_FOR_QA and not included.empty:
        included = included[included["curation_status"] == "auto_approved"].copy()
    gold_decks = pd.DataFrame([{"deck_id": deck_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "page_count": page_count, "pipeline_version": PIPELINE_VERSION, "loaded_ts": now_utc()}])
    ci_rows: List[Dict[str, Any]] = []
    if blocks_df is not None and not blocks_df.empty:
        ci = blocks_df[blocks_df["block_type"] == "ci_headline"].copy()
        for _, b in ci.iterrows():
            ci_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "headline_id": make_hash_id("headline", b.get("block_id")), "page_num": int(b.get("page_num")), "headline_type": b.get("section_id"), "lob": b.get("section_group"), "carrier": b.get("carrier"), "headline_rank": safe_float(b.get("category")), "headline_text": b.get("block_text"), "raw_evidence": b.get("block_text"), "curation_status": "system_extracted", "created_ts": now_utc()})
    gold_ci = pd.DataFrame(ci_rows)
    major = included[included["section_id"] == "major_offer_changes"].copy() if not included.empty else pd.DataFrame()
    major_rows: List[Dict[str, Any]] = []
    for _, o in major.iterrows():
        major_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "change_id": make_hash_id("change", o.get("offer_id")), "offer_id": o.get("offer_id"), "candidate_id": o.get("candidate_id"), "page_num": int(o.get("page_num")), "lob": o.get("section_group"), "carrier": o.get("carrier"), "change_type": o.get("offer_type"), "product_name": o.get("product_name"), "offer_summary": o.get("offer_summary"), "primary_value_text": o.get("primary_value_text"), "primary_value_num": o.get("primary_value_num"), "effective_start_text": None, "effective_end_text": None, "condition_text": o.get("condition_text"), "raw_evidence": o.get("raw_evidence"), "curation_status": o.get("curation_status"), "created_ts": now_utc()})
    gold_major = pd.DataFrame(major_rows)
    offers = included[included["section_id"] != "major_offer_changes"].copy() if not included.empty else pd.DataFrame()
    if not offers.empty:
        offers["lob"] = offers["section_group"]
        gold_offers = offers[["deck_id", "run_id", "pdf_hash", "pdf_name", "deck_week", "offer_id", "candidate_id", "page_num", "section_id", "section_group", "lob", "carrier", "brand", "product_name", "product_category", "offer_type", "offer_summary", "primary_value_text", "primary_value_num", "currency", "rate_plan_requirement", "customer_segment", "action_required", "condition_text", "date_start", "date_end", "raw_evidence", "confidence", "curation_status", "created_ts"]].copy()
    else:
        gold_offers = pd.DataFrame()
    if offer_values_df is not None and not offer_values_df.empty:
        ov = offer_values_df.copy()
        offer_status = normalized_df[["candidate_id", "offer_id", "curation_status"]].drop_duplicates() if normalized_df is not None and not normalized_df.empty else pd.DataFrame()
        ov = ov.merge(offer_status, on="candidate_id", how="left")
        gold_values = ov[["deck_id", "run_id", "pdf_hash", "pdf_name", "deck_week", "offer_value_id", "offer_id", "candidate_id", "page_num", "section_id", "value_role", "value_type", "value_text", "value_num", "value_num_min", "value_num_max", "currency", "unit", "period", "comparator", "raw_evidence", "curation_status", "created_ts"]].copy()
    else:
        gold_values = pd.DataFrame()
    if bridge_df is not None and not bridge_df.empty:
        b = bridge_df.copy()
        status = normalized_df[["offer_id", "curation_status"]].drop_duplicates() if normalized_df is not None and not normalized_df.empty else pd.DataFrame()
        b = b.merge(status, on="offer_id", how="left")
        gold_bridge = b[["deck_id", "run_id", "pdf_hash", "pdf_name", "deck_week", "bridge_id", "offer_id", "candidate_id", "page_num", "section_id", "carrier", "device_name", "device_brand", "device_family", "device_model", "curation_status", "created_ts"]].copy()
    else:
        gold_bridge = pd.DataFrame()
    gold_grid = price_rows_df.copy() if price_rows_df is not None and not price_rows_df.empty else pd.DataFrame()
    if not gold_grid.empty:
        gold_grid["curation_status"] = np.where(gold_grid["needs_review"] == True, "pending_review", "system_extracted")
        gold_grid = gold_grid[["deck_id", "run_id", "pdf_hash", "pdf_name", "deck_week", "grid_row_id", "page_num", "section_id", "section_group", "brand", "carrier", "device_name", "device_category", "retail_price", "customer_scenario", "id_requirement", "rate_plan_requirement", "price_text", "price_value", "savings_text", "savings_value", "raw_evidence", "confidence", "curation_status", "created_ts"]].copy()
    retail_rows: List[Dict[str, Any]] = []
    if blocks_df is not None and not blocks_df.empty:
        retail_blocks = blocks_df[blocks_df["block_type"] == "retail_readout_line"].copy()
        for _, r in retail_blocks.iterrows():
            val = safe_float(r.get("block_text"))
            retail_rows.append({"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "readout_id": make_hash_id("readout", r.get("block_id")), "page_num": int(r.get("page_num")), "retailer": infer_carrier_from_text(str(r.get("block_text"))) or None, "brand_or_carrier": r.get("carrier"), "device_name": extract_product_name(str(r.get("block_text"))), "promo_summary": r.get("block_text"), "promo_value_text": r.get("block_text") if val is not None else None, "promo_value_num": val, "source_context": "national_retail_readout", "raw_evidence": r.get("block_text"), "curation_status": "pending_review", "created_ts": now_utc()})
    gold_retail = pd.DataFrame(retail_rows)
    result = {"gold_decks": gold_decks, "gold_ciHeadlines": gold_ci, "gold_majorOfferChanges": gold_major, "gold_offers": gold_offers, "gold_offerValues": gold_values, "gold_offerDeviceBridge": gold_bridge, "gold_devicePriceGrid": gold_grid, "gold_retailReadout": gold_retail}
    for key, df in result.items():
        print(f"{key}: {len(df):,} rows")
    if PRINT_GOLD_EXAMPLES:
        for key, df in result.items():
            if df is not None and not df.empty:
                print_subheader(f"Sample {key}")
                show_df(df, max_rows=5)
    return result

# ======================================================================================
# SECTION 10 - BIGQUERY LOAD AND MAIN PIPELINE RUNNER
# ======================================================================================

def load_all_outputs_to_bq(outputs: Dict[str, pd.DataFrame]) -> None:
    if not WRITE_TO_BQ:
        print("WRITE_TO_BQ is False. Skipping BigQuery load.")
        return
    print_header("BIGQUERY LOAD")
    load_order = [
        "bronze_pdfDecks", "bronze_slidePages", "bronze_slideLines", "bronze_slideSpans",
        "bronze_slideRegions", "bronze_slideDetectedTables", "bronze_slideTableCells",
        "bronze_valueMentions", "bronze_agentExtractionsRaw",
        "silver_tocPageMap", "silver_slideSections", "silver_slideContentBlocks",
        "silver_offerCandidates", "silver_offerValues", "silver_normalizedOffers",
        "silver_offerDeviceBridge", "silver_priceGridRows",
        "review_extractionReview", "review_reviewDecisions",
        "gold_decks", "gold_ciHeadlines", "gold_majorOfferChanges", "gold_offers",
        "gold_offerValues", "gold_offerDeviceBridge", "gold_devicePriceGrid", "gold_retailReadout",
    ]
    for key in load_order:
        append_df_to_bq(outputs.get(key), key)


def main() -> Dict[str, pd.DataFrame]:
    print_header("STARTING COMPETITIVE OFFER REPORT PIPELINE V3")
    if WRITE_TO_BQ:
        ensure_dataset_and_tables()
    pdf_path = choose_pdf_path()
    pdf_hash = compute_file_hash(pdf_path)
    pdf_name = os.path.basename(pdf_path)
    temp_doc = fitz.open(pdf_path)
    page_count = temp_doc.page_count
    first_page_text = temp_doc[0].get_text("text") if page_count else ""
    temp_doc.close()
    deck_week = infer_deck_week_from_text_or_name(pdf_path, first_page_text)
    deck_id = f"deck_{deck_week.strftime('%Y%m%d')}_{pdf_hash[:12]}"
    run_id = f"run_{uuid.uuid4().hex[:16]}"
    print(f"pdf_name: {pdf_name}")
    print(f"pdf_hash: {pdf_hash}")
    print(f"deck_week: {deck_week}")
    print(f"deck_id: {deck_id}")
    print(f"run_id: {run_id}")
    print(f"page_count: {page_count}")

    duplicate_action = "new"
    existing = get_existing_decks_by_hash(pdf_hash)
    if not existing.empty:
        duplicate_action = resolve_duplicate_action(existing)
        if duplicate_action == "replace":
            delete_existing_rows_for_pdf_hash(pdf_hash)

    pdf_decks_df = pd.DataFrame([{"deck_id": deck_id, "run_id": run_id, "pdf_hash": pdf_hash, "pdf_name": pdf_name, "deck_week": deck_week, "pdf_path": pdf_path, "page_count": page_count, "file_size_bytes": os.path.getsize(pdf_path), "duplicate_action": duplicate_action, "pipeline_version": PIPELINE_VERSION, "ingestion_ts": now_utc()}])

    pages_df, lines_df, spans_df = extract_pdf_to_bronze(pdf_path, deck_id, run_id, pdf_hash, deck_week)
    toc_df = parse_toc_page_map(pages_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    sections_df = classify_all_slides(pages_df, lines_df, toc_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)

    # Update page table with title/header after classification.
    title_lookup = sections_df.set_index("page_num")["title_header_text"].to_dict()
    pages_df["title_header_text"] = pages_df["page_num"].map(title_lookup)

    regions_df, detected_tables_df = build_bronze_regions_and_tables(pages_df, sections_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    blocks_df = build_content_blocks(pages_df, sections_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    agent_raw_df, cells_df = run_gemini_vision_cell_extraction(pages_df, sections_df, detected_tables_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    value_mentions_df = build_bronze_value_mentions(blocks_df, cells_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    candidates_df, price_rows_df = build_offer_candidates(blocks_df, cells_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    offer_values_df = build_silver_offer_values(candidates_df, value_mentions_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    normalized_df = normalize_offers(candidates_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    bridge_df = build_offer_device_bridge(normalized_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    review_df, decisions_df = build_review_tables(candidates_df, price_rows_df, normalized_df, deck_id, run_id, pdf_hash, pdf_name, deck_week)
    gold_outputs = build_gold_tables(pdf_decks_df, blocks_df, normalized_df, offer_values_df, bridge_df, price_rows_df, deck_id, run_id, pdf_hash, pdf_name, deck_week, page_count)

    outputs: Dict[str, pd.DataFrame] = {
        "bronze_pdfDecks": pdf_decks_df,
        "bronze_slidePages": pages_df,
        "bronze_slideLines": lines_df,
        "bronze_slideSpans": spans_df,
        "bronze_slideRegions": regions_df,
        "bronze_slideDetectedTables": detected_tables_df,
        "bronze_slideTableCells": cells_df,
        "bronze_valueMentions": value_mentions_df,
        "bronze_agentExtractionsRaw": agent_raw_df,
        "silver_tocPageMap": toc_df,
        "silver_slideSections": sections_df,
        "silver_slideContentBlocks": blocks_df,
        "silver_offerCandidates": candidates_df,
        "silver_offerValues": offer_values_df,
        "silver_normalizedOffers": normalized_df,
        "silver_offerDeviceBridge": bridge_df,
        "silver_priceGridRows": price_rows_df,
        "review_extractionReview": review_df,
        "review_reviewDecisions": decisions_df,
    }
    outputs.update(gold_outputs)
    load_all_outputs_to_bq(outputs)

    print_header("FINAL RUN SUMMARY")
    summary_rows = []
    for key, df in outputs.items():
        summary_rows.append({"table_key": key, "rows": 0 if df is None else len(df)})
    summary_df = pd.DataFrame(summary_rows)
    show_df(summary_df, max_rows=100)

    if PRINT_SAMPLE_OUTPUTS:
        print_header("KEY QA SAMPLES")
        if not sections_df.empty:
            print_subheader("Slide routing")
            show_df(sections_df[["page_num", "section_id", "section_group", "layout_type", "extraction_route", "toc_mismatch_flag"]], 50)
        if cells_df is not None and not cells_df.empty:
            print_subheader("Bronze table cells")
            show_df(cells_df[["page_num", "section_id", "table_type", "row_header", "column_header", "carrier_context", "product_context", "cell_text", "confidence", "needs_review"]], 20)
        if candidates_df is not None and not candidates_df.empty:
            print_subheader("Silver offer candidates")
            show_df(candidates_df[["page_num", "section_id", "carrier", "product_name", "offer_type", "offer_amount_text", "confidence", "needs_review", "raw_evidence"]], 20)
        if price_rows_df is not None and not price_rows_df.empty:
            print_subheader("Silver price grid rows")
            show_df(price_rows_df[["page_num", "section_id", "brand", "device_name", "customer_scenario", "rate_plan_requirement", "price_text", "price_value", "confidence", "needs_review"]], 20)
    print_header("PIPELINE COMPLETE")
    return outputs


# ======================================================================================
# SECTION 11 - RUN THE PIPELINE
# ======================================================================================

pipeline_outputs = main()
